# NYC 311 Schema and Data-Quality Analysis

## Day 3 objective

Inspect schema, missing values, duplicate complaint identifiers, and
timestamp consistency for the **complete live NYC 311 dataset**.

The Month 1 business question is whether a newly created complaint will miss
its expected resolution target. The initial target concept compares
`closed_date` with `due_date`. This notebook assesses the source fields and
their quality; it does **not** create the target, clean records, impute values,
remove duplicates, choose final features, split data, or train a model.

All material counts and decisions come from server-side SoQL aggregations over
the complete live source. Small row-level queries, when present, are labelled
as examples and never estimate full-dataset quality.


## 1. Imports and project paths

> **Analysis scope:** Complete live NYC 311 dataset using server-side SoQL
> aggregation.

The notebook reuses the repository's API configuration and report-path
helpers. Requests are read-only, use explicit timeouts and bounded retries,
and retain the secure DNS fallback established in the Day 2 notebook.


In [1]:
from datetime import datetime, timezone
from http.client import HTTPSConnection, RemoteDisconnected
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode, urlsplit
from urllib.request import Request, urlopen
import json
import socket
import sys
import time

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 160)

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "src" / "urban_ops").is_dir()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from urban_ops.data.api_client import fetch_json_url
from urban_ops.data.nyc_311_config import (
    API_ENDPOINT,
    API_MAX_ATTEMPTS,
    API_TIMEOUT_SECONDS,
    SELECTED_COLUMNS,
)
from urban_ops.utils.paths import ensure_report_directories, notebook_report_paths

NOTEBOOK_SLUG = "02_schema_and_quality_analysis"
REPORT_DIR, REPORT_TABLES_DIR, REPORT_FIGURES_DIR = notebook_report_paths(
    NOTEBOOK_SLUG
)
ensure_report_directories(NOTEBOOK_SLUG)

DATASET_ID = API_ENDPOINT.rsplit("/", maxsplit=1)[-1].removesuffix(".json")
DATASET_METADATA_URL = f"https://data.cityofnewyork.us/api/views/{DATASET_ID}"
QUALITY_REPORT_PATH = REPORT_DIR / "data_quality_report.md"

# Remove names emitted by the superseded sample-based Day 3 implementation.
for obsolete_output_name in [
    "data_quality_issue_register.csv",
    "schema_validation_summary.csv",
]:
    (REPORT_TABLES_DIR / obsolete_output_name).unlink(missing_ok=True)

PROJECT_ROOT, DATASET_ID, QUALITY_REPORT_PATH.relative_to(PROJECT_ROOT)


(PosixPath('/Users/mohammadmubashir/VCode/urban-operations-intelligence-platform'),
 'erm2-nwe9',
 PosixPath('reports/02_schema_and_quality_analysis/data_quality_report.md'))

## 2. Column roles, quality thresholds, and analysis configuration

> **Analysis scope:** Complete live NYC 311 dataset using server-side SoQL
> aggregation.

The selected fields reflect the current business problem and pipeline needs;
they are not a universal list of equally mandatory columns.

- Core fields provide minimum complaint-level structure.
- `closed_date` and `due_date` are mathematically essential to the initial
  target concept.
- `status` is separate: it supports eligibility decisions and contradiction
  checks, but is not part of `closed_date > due_date`.
- Candidate features are possibilities only. They require later checks for
  creation-time availability, leakage, missingness, stability, and predictive
  value.
- Supporting metadata improves interpretation and consistency checks.

`closed_date` is an outcome timestamp. Final `status` and other post-creation
fields are leakage risks and must not be used as creation-time model features.
Final feature selection and target eligibility belong to later notebooks.


In [2]:
CORE_REQUIRED_COLUMNS = [
    "unique_key",
    "created_date",
    "agency",
    "complaint_type",
]

ESSENTIAL_TARGET_COLUMNS = [
    "closed_date",
    "due_date",
]

TARGET_ELIGIBILITY_AND_VALIDATION_COLUMNS = [
    "status",
]

CANDIDATE_FEATURE_COLUMNS = [
    "agency",
    "complaint_type",
    "descriptor",
    "borough",
    "open_data_channel_type",
]

SUPPORTING_COLUMNS = [
    "agency_name",
]

TIMESTAMP_COLUMNS = [
    "created_date",
    "closed_date",
    "due_date",
    "resolution_action_updated_date",
]

CATEGORICAL_QUALITY_COLUMNS = [
    "agency",
    "agency_name",
    "complaint_type",
    "descriptor",
    "status",
    "borough",
    "open_data_channel_type",
]

GEOGRAPHIC_QUALITY_COLUMNS = [
    "latitude",
    "longitude",
    "incident_zip",
    "borough",
]

MISSINGNESS_THRESHOLDS = {
    "low_upper_exclusive": 5.0,
    "moderate_upper_exclusive": 30.0,
}

PLACEHOLDER_VALUES = {
    "unknown",
    "n/a",
    "na",
    "unspecified",
    "other",
}

APPROXIMATE_NYC_BOUNDS = {
    "minimum_latitude": 40.45,
    "maximum_latitude": 40.95,
    "minimum_longitude": -74.30,
    "maximum_longitude": -73.65,
}

GROUP_PAGE_SIZE = 50_000
MISSINGNESS_QUERY_CHUNK_SIZE = 12
EXACT_DUPLICATE_TIMEOUT_SECONDS = 90
ANALYSIS_MODE = "full-dataset server-side aggregation"
QUALITY_METRICS_USE_SAMPLE = "no"

COLUMN_ROLE_RECORDS = [
    {
        "column_name": "unique_key",
        "column_group": "core dataset requirement",
        "role": "identifier",
        "why_needed": "Duplicate detection and complaint-level traceability.",
        "required_for_current_stage": "yes",
        "available_at_prediction_time": "yes",
        "potential_leakage": "no",
    },
    {
        "column_name": "created_date",
        "column_group": "core dataset requirement",
        "role": "event timestamp",
        "why_needed": "Prediction moment, timestamp validation, and chronological splitting.",
        "required_for_current_stage": "yes",
        "available_at_prediction_time": "yes",
        "potential_leakage": "no",
    },
    {
        "column_name": "closed_date",
        "column_group": "essential target input",
        "role": "actual outcome timestamp",
        "why_needed": "Represents actual closure time for comparison with the expected deadline.",
        "required_for_current_stage": "yes, for target-input quality analysis",
        "available_at_prediction_time": "no",
        "potential_leakage": "yes, outcome information",
    },
    {
        "column_name": "due_date",
        "column_group": "essential target input",
        "role": "expected resolution deadline",
        "why_needed": "Represents the deadline used by the initial target concept.",
        "required_for_current_stage": "yes, for target-input quality analysis",
        "available_at_prediction_time": "must be confirmed",
        "potential_leakage": "depends on whether it exists at complaint creation",
    },
    {
        "column_name": "status",
        "column_group": "target eligibility and validation",
        "role": "workflow state",
        "why_needed": "Supports eligibility rules and detects closure/timestamp contradictions.",
        "required_for_current_stage": "yes, for target quality validation",
        "available_at_prediction_time": "initial state may be available; final state is not",
        "potential_leakage": "yes, when final status is used",
    },
    {
        "column_name": "agency",
        "column_group": "core requirement and candidate feature",
        "role": "responsible organization",
        "why_needed": "Scope analysis, subgroup quality analysis, and possible modelling.",
        "required_for_current_stage": "yes",
        "available_at_prediction_time": "yes",
        "potential_leakage": "no",
    },
    {
        "column_name": "agency_name",
        "column_group": "supporting metadata",
        "role": "human-readable agency label",
        "why_needed": "Agency-code mapping validation and reporting.",
        "required_for_current_stage": "no",
        "available_at_prediction_time": "usually yes",
        "potential_leakage": "no",
    },
    {
        "column_name": "complaint_type",
        "column_group": "core requirement and candidate feature",
        "role": "main request category",
        "why_needed": "Scope analysis, category baselines, and possible modelling.",
        "required_for_current_stage": "yes",
        "available_at_prediction_time": "yes",
        "potential_leakage": "no",
    },
    {
        "column_name": "descriptor",
        "column_group": "candidate feature",
        "role": "detailed complaint category",
        "why_needed": "Additional operational detail and possible categorical modelling.",
        "required_for_current_stage": "no",
        "available_at_prediction_time": "generally yes; confirm source timing",
        "potential_leakage": "no known leakage; validate later",
    },
    {
        "column_name": "borough",
        "column_group": "candidate feature",
        "role": "geographic grouping",
        "why_needed": "Subgroup analysis and location-related modelling.",
        "required_for_current_stage": "no",
        "available_at_prediction_time": "generally yes; confirm source timing",
        "potential_leakage": "no known leakage; validate later",
    },
    {
        "column_name": "open_data_channel_type",
        "column_group": "candidate feature",
        "role": "complaint intake channel",
        "why_needed": "Quality analysis and possible creation-time signal.",
        "required_for_current_stage": "no",
        "available_at_prediction_time": "yes",
        "potential_leakage": "no",
    },
]

column_role_summary = pd.DataFrame(COLUMN_ROLE_RECORDS)
column_role_summary.to_csv(
    REPORT_TABLES_DIR / "column_role_summary.csv",
    index=False,
)
column_role_summary


,column_name,column_group,role,why_needed,required_for_current_stage,available_at_prediction_time,potential_leakage
0,unique_key,core dataset requirement,identifier,Duplicate detection and complaint-level traceability.,yes,yes,no
1,created_date,core dataset requirement,event timestamp,"Prediction moment, timestamp validation, and chronological splitting.",yes,yes,no
2,closed_date,essential target input,actual outcome timestamp,Represents actual closure time for comparison with the expected deadline.,"yes, for target-input quality analysis",no,"yes, outcome information"
3,due_date,essential target input,expected resolution deadline,Represents the deadline used by the initial target concept.,"yes, for target-input quality analysis",must be confirmed,depends on whether it exists at complaint creation
4,status,target eligibility and validation,workflow state,Supports eligibility rules and detects closure/timestamp contradictions.,"yes, for target quality validation",initial state may be available; final state is not,"yes, when final status is used"
5,agency,core requirement and candidate feature,responsible organization,"Scope analysis, subgroup quality analysis, and possible modelling.",yes,yes,no
6,agency_name,supporting metadata,human-readable agency label,Agency-code mapping validation and reporting.,no,usually yes,no
7,complaint_type,core requirement and candidate feature,main request category,"Scope analysis, category baselines, and possible modelling.",yes,yes,no
8,descriptor,candidate feature,detailed complaint category,Additional operational detail and possible categorical modelling.,no,generally yes; confirm source timing,no known leakage; validate later
9,borough,candidate feature,geographic grouping,Subgroup analysis and location-related modelling.,no,generally yes; confirm source timing,no known leakage; validate later


## 3. Audited read-only API helpers

> **Analysis scope:** Complete live NYC 311 dataset using server-side SoQL
> aggregation.

Every SoQL request is logged with its query text, execution status, retrieved
row count, UTC timestamp, and any error. Grouped results use stable ordering
and bounded pagination. Failures are either raised or returned explicitly to
the caller; they are never silently converted into empty data or zero counts.

The lightweight retry and DNS handling here support reproducible exploration.
They are not the future production ingestion implementation under `src/data/`.


In [3]:
QUERY_AUDIT_RECORDS: list[dict[str, object]] = []
QUERY_FAILURES: list[dict[str, str]] = []
API_REQUEST_HEADERS = {
    "User-Agent": "urban-operations-intelligence-day3/1.0",
}


def exception_chain_text(error: BaseException) -> str:
    """Return concise messages for an exception and its nested causes."""
    messages = []
    current_error: BaseException | None = error
    visited_error_ids: set[int] = set()
    while current_error is not None and id(current_error) not in visited_error_ids:
        visited_error_ids.add(id(current_error))
        message = f"{type(current_error).__name__}: {current_error}"
        if message not in messages:
            messages.append(message)
        current_error = current_error.__cause__ or current_error.__context__
    return " <- ".join(messages)


def record_query_audit(
    query_name: str,
    query: str,
    execution_status: str,
    retrieved_rows: int | None,
    error_message: str,
) -> None:
    """Append one immutable query-execution audit record."""
    QUERY_AUDIT_RECORDS.append(
        {
            "query_name": query_name,
            "query": query,
            "execution_status": execution_status,
            "retrieved_rows": retrieved_rows,
            "executed_at_utc": datetime.now(timezone.utc).isoformat(),
            "error_message": error_message,
        }
    )


def execute_soql(
    query_name: str,
    query: str,
    *,
    allow_failure: bool = False,
    timeout_seconds: float = API_TIMEOUT_SECONDS,
    max_attempts: int = API_MAX_ATTEMPTS,
) -> list[dict[str, object]] | None:
    """Execute one audited SoQL query and never disguise a failure as zero."""
    query_url = f"{API_ENDPOINT}?{urlencode({'$query': query})}"
    try:
        payload = fetch_json_url(
            query_url,
            timeout_seconds=timeout_seconds,
            max_attempts=max_attempts,
        )
        if not isinstance(payload, list):
            raise RuntimeError("NYC Open Data returned JSON that was not a list.")
        records = [
            record for record in payload if isinstance(record, dict)
        ]
        if len(records) != len(payload):
            raise RuntimeError("NYC Open Data returned a non-object row.")
        record_query_audit(query_name, query, "success", len(records), "")
        return records
    except Exception as error:
        error_message = exception_chain_text(error)
        record_query_audit(query_name, query, "failed", None, error_message)
        QUERY_FAILURES.append(
            {
                "query_name": query_name,
                "reason": error_message,
            }
        )
        if allow_failure:
            return None
        raise RuntimeError(f"SoQL query '{query_name}' failed.") from error


def fetch_single_aggregate(
    query_name: str,
    query: str,
    *,
    allow_failure: bool = False,
    timeout_seconds: float = API_TIMEOUT_SECONDS,
    max_attempts: int = API_MAX_ATTEMPTS,
) -> dict[str, object] | None:
    """Return the sole record produced by an aggregate query."""
    records = execute_soql(
        query_name,
        query,
        allow_failure=allow_failure,
        timeout_seconds=timeout_seconds,
        max_attempts=max_attempts,
    )
    if records is None:
        return None
    if len(records) != 1:
        raise RuntimeError(
            f"Aggregate query '{query_name}' returned {len(records)} rows."
        )
    return records[0]


def fetch_paginated_grouped(
    query_name: str,
    query_without_paging: str,
    output_columns: list[str],
    *,
    allow_failure: bool = False,
    page_size: int = GROUP_PAGE_SIZE,
    timeout_seconds: float = API_TIMEOUT_SECONDS,
) -> pd.DataFrame | None:
    """Retrieve every deterministically ordered grouped row by pagination."""
    all_records: list[dict[str, object]] = []
    offset = 0
    page_number = 1
    while True:
        paged_query = (
            f"{query_without_paging} LIMIT {page_size} OFFSET {offset}"
        )
        records = execute_soql(
            f"{query_name}_page_{page_number:04d}",
            paged_query,
            allow_failure=allow_failure,
            timeout_seconds=timeout_seconds,
        )
        if records is None:
            return None
        all_records.extend(records)
        if len(records) < page_size:
            break
        offset += page_size
        page_number += 1
    return pd.DataFrame(all_records).reindex(columns=output_columns)


def conditional_count_expression(condition: str, alias: str) -> str:
    """Return a SoQL conditional sum that always produces a numeric count."""
    return f"sum(case({condition}, 1, true, 0)) AS {alias}"


def safe_int(value: object) -> int | None:
    """Convert a numeric API value to int while preserving missing results."""
    if value is None or value == "":
        return None
    return int(value)


def safe_float(value: object) -> float | None:
    """Convert a numeric API value to float while preserving missing results."""
    if value is None or value == "":
        return None
    return float(value)


def percentage(count: int | None, total: int | None) -> float | None:
    """Calculate a percentage without turning unavailable results into zero."""
    if count is None or total in (None, 0):
        return None
    return round(100.0 * count / total, 6)


def soql_string_list(values: set[str]) -> str:
    """Return safely quoted deterministic SoQL string literals."""
    quoted = [
        "'" + value.replace("'", "''") + "'"
        for value in sorted(values)
    ]
    return "(" + ", ".join(quoted) + ")"


## 4. Source scope, API metadata, and authoritative schema

> **Analysis scope:** Complete live NYC 311 dataset using server-side SoQL
> aggregation.

The NYC Open Data metadata endpoint is authoritative for source types. A
sample dataframe's inferred pandas dtypes are not used as schema evidence.
Identifiers retain identifier semantics even when a source happens to encode
them as text or numeric values.


In [4]:
analysis_timestamp_utc = datetime.now(timezone.utc)
metadata_payload = fetch_json_url(
    DATASET_METADATA_URL,
    timeout_seconds=API_TIMEOUT_SECONDS,
    max_attempts=API_MAX_ATTEMPTS,
    headers=API_REQUEST_HEADERS,
)
if not isinstance(metadata_payload, dict):
    raise RuntimeError("Dataset metadata response was not a JSON object.")

metadata_columns = metadata_payload.get("columns")
if not isinstance(metadata_columns, list):
    raise RuntimeError("Dataset metadata did not include a column list.")

metadata_by_field = {
    str(column["fieldName"]): column
    for column in metadata_columns
    if isinstance(column, dict) and column.get("fieldName")
}
dataset_columns = set(metadata_by_field)

scope_record = fetch_single_aggregate(
    "dataset_scope",
    (
        "SELECT count(*) AS total_rows, "
        "min(created_date) AS minimum_created_date, "
        "max(created_date) AS maximum_created_date"
    ),
)
if scope_record is None:
    raise RuntimeError("Dataset scope query did not return a result.")

total_row_count = safe_int(scope_record.get("total_rows"))
if total_row_count is None:
    raise RuntimeError("Dataset scope query did not return total_rows.")

minimum_created_date = scope_record.get("minimum_created_date")
maximum_created_date = scope_record.get("maximum_created_date")

quality_analysis_scope = pd.DataFrame(
    [
        {
            "source": metadata_payload.get("name", "NYC Open Data"),
            "api_endpoint": API_ENDPOINT,
            "dataset_identifier": DATASET_ID,
            "retrieval_timestamp_utc": analysis_timestamp_utc.isoformat(),
            "total_dataset_rows": total_row_count,
            "minimum_created_date": minimum_created_date,
            "maximum_created_date": maximum_created_date,
            "analysis_mode": ANALYSIS_MODE,
            "sample_used_for_quality_metrics": QUALITY_METRICS_USE_SAMPLE,
        }
    ]
)
quality_analysis_scope.to_csv(
    REPORT_TABLES_DIR / "quality_analysis_scope.csv",
    index=False,
)


def logical_type_for(column_name: str, source_data_type: str) -> str:
    """Map authoritative source types to project-level logical semantics."""
    if column_name == "unique_key":
        return "identifier"
    if source_data_type == "calendar_date":
        return "datetime"
    if source_data_type == "number":
        return "numeric"
    if source_data_type == "point":
        return "geospatial_point"
    return "categorical_or_text"


schema_rows = []
for column_name in SELECTED_COLUMNS:
    metadata = metadata_by_field.get(column_name, {})
    source_data_type = str(metadata.get("dataTypeName", "unavailable"))
    schema_rows.append(
        {
            "column_name": column_name,
            "source_data_type": source_data_type,
            "logical_type": logical_type_for(column_name, source_data_type),
            "source_description": str(metadata.get("description", "")).strip(),
            "is_core_required": column_name in CORE_REQUIRED_COLUMNS,
            "is_essential_target": column_name in ESSENTIAL_TARGET_COLUMNS,
            "is_target_eligibility_field": (
                column_name in TARGET_ELIGIBILITY_AND_VALIDATION_COLUMNS
            ),
            "is_candidate_feature": column_name in CANDIDATE_FEATURE_COLUMNS,
            "is_supporting_metadata": column_name in SUPPORTING_COLUMNS,
            "available_in_source": column_name in dataset_columns,
        }
    )

schema_summary = pd.DataFrame(schema_rows)
schema_summary.to_csv(
    REPORT_TABLES_DIR / "schema_summary.csv",
    index=False,
)

display(quality_analysis_scope)
display(schema_summary)


,source,api_endpoint,dataset_identifier,retrieval_timestamp_utc,total_dataset_rows,minimum_created_date,maximum_created_date,analysis_mode,sample_used_for_quality_metrics
0,311 Service Requests from 2020 to Present,https://data.cityofnewyork.us/resource/erm2-nwe9.json,erm2-nwe9,2026-07-25T08:06:29.042177+00:00,21922301,2020-01-01T00:00:00.000,2026-07-24T01:50:58.000,full-dataset server-side aggregation,no


,column_name,source_data_type,logical_type,source_description,is_core_required,is_essential_target,is_target_eligibility_field,is_candidate_feature,is_supporting_metadata,available_in_source
0,unique_key,text,identifier,Unique identifier of a Service Request (SR) in the open data set,True,False,False,False,False,True
1,created_date,calendar_date,datetime,Date SR was created,True,False,False,False,False,True
2,closed_date,calendar_date,datetime,Date SR was closed by responding agency,False,True,False,False,False,True
3,agency,text,categorical_or_text,Acronym of responding City Government Agency,True,False,False,True,False,True
4,agency_name,text,categorical_or_text,Full Agency name of responding City Government Agency,False,False,False,False,True,True
5,complaint_type,text,categorical_or_text,This is the first level of a hierarchy identifying the topic of the incident or condition. Problem (formerly Complaint Type) broadly describes the topic of...,True,False,False,True,False,True
6,descriptor,text,categorical_or_text,"This is associated to the Problem (formerly Complaint Type), and provides further detail on the incident or condition. Problem Detail (formerly Descriptor) ...",False,False,False,True,False,True
7,descriptor_2,text,categorical_or_text,A third level of detail about the Problem (formerly Complaint Type) beyond the Problem Detail (formerly Descriptor). This is not used by every category of i...,False,False,False,False,False,True
8,location_type,text,categorical_or_text,Describes the type of location used in the address information,False,False,False,False,False,True
9,incident_zip,text,categorical_or_text,"Incident location zip code, provided by geo validation.",False,False,False,False,False,True


## 5. Requirement-aware schema validation

> **Analysis scope:** Complete live NYC 311 dataset using authoritative API
> metadata and server-side SoQL aggregation.

Missing fields have different consequences. Essential target inputs block the
initial target formula, while a missing `status` limits eligibility and
contradiction analysis without changing the formula itself. A missing
candidate feature does not invalidate the core schema.


In [5]:
GROUP_VALIDATION_RULES = {
    "core_required": {
        "columns": CORE_REQUIRED_COLUMNS,
        "severity_if_missing": "critical",
        "impact_if_missing": (
            "The source cannot reliably support complaint-level analysis."
        ),
    },
    "essential_target": {
        "columns": ESSENTIAL_TARGET_COLUMNS,
        "severity_if_missing": "critical_for_target",
        "impact_if_missing": (
            "General NYC 311 analysis may remain possible, but the current "
            "missed-resolution-target definition cannot be evaluated."
        ),
    },
    "target_eligibility_and_validation": {
        "columns": TARGET_ELIGIBILITY_AND_VALIDATION_COLUMNS,
        "severity_if_missing": "warning",
        "impact_if_missing": (
            "Target eligibility and status/timestamp contradiction checks are limited."
        ),
    },
    "candidate_feature": {
        "columns": CANDIDATE_FEATURE_COLUMNS,
        "severity_if_missing": "warning",
        "impact_if_missing": (
            "This specific future modelling or subgroup-analysis option is reduced."
        ),
    },
    "supporting_metadata": {
        "columns": SUPPORTING_COLUMNS,
        "severity_if_missing": "informational",
        "impact_if_missing": "Reporting or consistency-check capability is reduced.",
    },
}


def validate_column_groups(
    available_columns: set[str],
    group_rules: dict[str, dict[str, object]],
) -> pd.DataFrame:
    """Return one validation row for each classified column membership."""
    rows = []
    for column_group, rule in group_rules.items():
        for column_name in rule["columns"]:
            rows.append(
                {
                    "column_name": column_name,
                    "column_group": column_group,
                    "is_present": column_name in available_columns,
                    "severity_if_missing": rule["severity_if_missing"],
                    "impact_if_missing": rule["impact_if_missing"],
                }
            )
    return pd.DataFrame(rows)


column_requirement_validation = validate_column_groups(
    dataset_columns,
    GROUP_VALIDATION_RULES,
)
column_requirement_validation.to_csv(
    REPORT_TABLES_DIR / "column_requirement_validation.csv",
    index=False,
)
column_requirement_validation


,column_name,column_group,is_present,severity_if_missing,impact_if_missing
0,unique_key,core_required,True,critical,The source cannot reliably support complaint-level analysis.
1,created_date,core_required,True,critical,The source cannot reliably support complaint-level analysis.
2,agency,core_required,True,critical,The source cannot reliably support complaint-level analysis.
3,complaint_type,core_required,True,critical,The source cannot reliably support complaint-level analysis.
4,closed_date,essential_target,True,critical_for_target,"General NYC 311 analysis may remain possible, but the current missed-resolution-target definition cannot be evaluated."
5,due_date,essential_target,True,critical_for_target,"General NYC 311 analysis may remain possible, but the current missed-resolution-target definition cannot be evaluated."
6,status,target_eligibility_and_validation,True,warning,Target eligibility and status/timestamp contradiction checks are limited.
7,agency,candidate_feature,True,warning,This specific future modelling or subgroup-analysis option is reduced.
8,complaint_type,candidate_feature,True,warning,This specific future modelling or subgroup-analysis option is reduced.
9,descriptor,candidate_feature,True,warning,This specific future modelling or subgroup-analysis option is reduced.


## 6. Full-dataset missing-value analysis

> **Analysis scope:** Complete live NYC 311 dataset using server-side SoQL
> aggregation.

For every configured project field, the API computes `count(*)` and
`count(column)`. Missingness classifications use the single named threshold
configuration above. No values are filled, removed, or changed.


In [6]:
def missingness_level(missing_percentage: float | None) -> str:
    """Classify missingness with the configured, named thresholds."""
    if missing_percentage is None:
        return "not_computable"
    if missing_percentage == 0:
        return "none"
    if missing_percentage < MISSINGNESS_THRESHOLDS["low_upper_exclusive"]:
        return "low"
    if missing_percentage < MISSINGNESS_THRESHOLDS["moderate_upper_exclusive"]:
        return "moderate"
    if missing_percentage < 100:
        return "high"
    return "complete"


def column_roles(column_name: str) -> str:
    """Return all requirement roles held by a configured column."""
    roles = []
    for group_name, rule in GROUP_VALIDATION_RULES.items():
        if column_name in rule["columns"]:
            roles.append(group_name)
    return ", ".join(roles) if roles else "selected_project_field"


def missingness_implication(
    column_name: str,
    missing_percentage: float | None,
) -> str:
    """Describe missingness without imposing cleaning or eligibility rules."""
    if missing_percentage is None:
        return "Missingness could not be computed because the source field is unavailable."
    if missing_percentage == 0:
        return "No null values detected in the complete source."
    if column_name in ESSENTIAL_TARGET_COLUMNS:
        return (
            "Coverage limits rows that could be evaluated under the initial target "
            "concept; final eligibility is deferred to Day 4."
        )
    if column_name in CORE_REQUIRED_COLUMNS:
        return "Missing values may impair complaint-level traceability or core grouping."
    if column_name in CANDIDATE_FEATURE_COLUMNS:
        return (
            "Coverage limits this candidate's future usefulness but does not invalidate "
            "the core dataset."
        )
    return "Coverage should be considered in later scoped ingestion and analysis."


available_selected_columns = [
    column for column in SELECTED_COLUMNS if column in dataset_columns
]
non_null_counts: dict[str, int] = {}

for chunk_index in range(
    0,
    len(available_selected_columns),
    MISSINGNESS_QUERY_CHUNK_SIZE,
):
    chunk = available_selected_columns[
        chunk_index : chunk_index + MISSINGNESS_QUERY_CHUNK_SIZE
    ]
    expressions = ["count(*) AS total_rows"] + [
        f"count({column}) AS {column}_non_null" for column in chunk
    ]
    record = fetch_single_aggregate(
        f"missingness_chunk_{chunk_index // MISSINGNESS_QUERY_CHUNK_SIZE + 1:02d}",
        "SELECT " + ", ".join(expressions),
    )
    if record is None:
        raise RuntimeError("A required missingness query did not return a result.")
    chunk_total = safe_int(record.get("total_rows"))
    if chunk_total != total_row_count:
        raise RuntimeError("Missingness query total does not match dataset scope.")
    for column in chunk:
        value = safe_int(record.get(f"{column}_non_null"))
        if value is None:
            raise RuntimeError(f"Missing count result for {column}.")
        non_null_counts[column] = value

missingness_rows = []
for column_name in SELECTED_COLUMNS:
    if column_name not in dataset_columns:
        non_null_count = None
        missing_count = None
        missing_percentage = None
    else:
        non_null_count = non_null_counts[column_name]
        missing_count = total_row_count - non_null_count
        missing_percentage = percentage(missing_count, total_row_count)
    missingness_rows.append(
        {
            "column_name": column_name,
            "total_rows": total_row_count,
            "non_null_count": non_null_count,
            "missing_count": missing_count,
            "missing_percentage": missing_percentage,
            "missingness_level": missingness_level(missing_percentage),
            "column_role": column_roles(column_name),
            "quality_implication": missingness_implication(
                column_name,
                missing_percentage,
            ),
        }
    )

missing_values_summary = (
    pd.DataFrame(missingness_rows)
    .sort_values(
        ["missing_percentage", "column_name"],
        ascending=[False, True],
        na_position="first",
    )
    .reset_index(drop=True)
)
missing_values_summary.to_csv(
    REPORT_TABLES_DIR / "missing_values_summary.csv",
    index=False,
)
missing_values_summary


,column_name,total_rows,non_null_count,missing_count,missing_percentage,missingness_level,column_role,quality_implication
0,taxi_company_borough,21922301,12681,21909620,99.942155,high,selected_project_field,Coverage should be considered in later scoped ingestion and analysis.
1,road_ramp,21922301,50865,21871436,99.767976,high,selected_project_field,Coverage should be considered in later scoped ingestion and analysis.
2,bridge_highway_direction,21922301,73117,21849184,99.666472,high,selected_project_field,Coverage should be considered in later scoped ingestion and analysis.
3,due_date,21922301,76987,21845314,99.648819,high,essential_target,Coverage limits rows that could be evaluated under the initial target concept; final eligibility is deferred to Day 4.
4,bridge_highway_name,21922301,135748,21786553,99.380777,high,selected_project_field,Coverage should be considered in later scoped ingestion and analysis.
5,bridge_highway_segment,21922301,135763,21786538,99.380708,high,selected_project_field,Coverage should be considered in later scoped ingestion and analysis.
6,taxi_pick_up_location,21922301,193493,21728808,99.117369,high,selected_project_field,Coverage should be considered in later scoped ingestion and analysis.
7,vehicle_type,21922301,441566,21480735,97.985768,high,selected_project_field,Coverage should be considered in later scoped ingestion and analysis.
8,facility_type,21922301,1673079,20249222,92.368141,high,selected_project_field,Coverage should be considered in later scoped ingestion and analysis.
9,descriptor_2,21922301,9635572,12286729,56.046712,high,selected_project_field,Coverage should be considered in later scoped ingestion and analysis.


## 7. Complete status vocabulary and transparent categories

> **Analysis scope:** Complete live NYC 311 dataset using server-side SoQL
> aggregation.

Status categories are defined only after inspecting the complete source
distribution. The sets below intersect explicit business labels with observed
values, so new statuses remain visible rather than being silently classified.
`status` supports eligibility and validation; it is not mathematically
essential to `closed_date > due_date`, and final status is a leakage field.


In [7]:
status_distribution = fetch_paginated_grouped(
    "status_distribution",
    (
        "SELECT status AS status, count(*) AS record_count "
        "GROUP BY status ORDER BY status"
    ),
    ["status", "record_count"],
)
if status_distribution is None:
    raise RuntimeError("The complete status distribution is required.")
status_distribution["record_count"] = pd.to_numeric(
    status_distribution["record_count"],
    errors="raise",
).astype("int64")

observed_statuses = {
    str(value)
    for value in status_distribution["status"].dropna().tolist()
}
CLOSED_STATUSES = observed_statuses & {"Closed"}
OPEN_OR_ACTIVE_STATUSES = observed_statuses & {
    "Assigned",
    "In Progress",
    "Open",
    "Pending",
    "Started",
}
EXCLUDED_OR_SPECIAL_STATUSES = observed_statuses & {
    "Cancel",
    "Cancelled",
    "Unspecified",
}
UNCLASSIFIED_STATUSES = (
    observed_statuses
    - CLOSED_STATUSES
    - OPEN_OR_ACTIVE_STATUSES
    - EXCLUDED_OR_SPECIAL_STATUSES
)

status_category_configuration = pd.DataFrame(
    [
        {
            "status_category": "closed",
            "observed_values": ", ".join(sorted(CLOSED_STATUSES)) or "None",
        },
        {
            "status_category": "open_or_active",
            "observed_values": ", ".join(sorted(OPEN_OR_ACTIVE_STATUSES)) or "None",
        },
        {
            "status_category": "excluded_or_special",
            "observed_values": (
                ", ".join(sorted(EXCLUDED_OR_SPECIAL_STATUSES)) or "None"
            ),
        },
        {
            "status_category": "unclassified",
            "observed_values": ", ".join(sorted(UNCLASSIFIED_STATUSES)) or "None",
        },
    ]
)
display(status_distribution)
display(status_category_configuration)


,status,record_count
0,Assigned,24714
1,Cancel,1
2,Closed,21482102
3,In Progress,260900
4,Open,82747
5,Pending,62962
6,Started,6093
7,Unspecified,2782


,status_category,observed_values
0,closed,Closed
1,open_or_active,"Assigned, In Progress, Open, Pending, Started"
2,excluded_or_special,"Cancel, Unspecified"
3,unclassified,None


## 8. Full-dataset missing-pattern checks

> **Analysis scope:** Complete live NYC 311 dataset using server-side SoQL
> aggregation.

Conditional sums count important combinations over all rows. These patterns
describe source quality and target-input coverage; they do not establish final
target eligibility rules.


In [8]:
closed_status_condition = (
    f"status IN {soql_string_list(CLOSED_STATUSES)}"
    if CLOSED_STATUSES
    else "false"
)
non_closed_observed_statuses = observed_statuses - CLOSED_STATUSES
non_closed_status_condition = (
    f"status IN {soql_string_list(non_closed_observed_statuses)}"
    if non_closed_observed_statuses
    else "false"
)

MISSING_PATTERN_RULES = [
    {
        "check_name": "closed_and_due_both_missing",
        "description": "closed_date missing and due_date missing",
        "condition": "closed_date IS NULL AND due_date IS NULL",
        "severity": "critical_for_target",
        "interpretation": "Neither essential input to the initial target concept is present.",
    },
    {
        "check_name": "closed_present_due_missing",
        "description": "closed_date present and due_date missing",
        "condition": "closed_date IS NOT NULL AND due_date IS NULL",
        "severity": "critical_for_target",
        "interpretation": "An actual closure exists without the expected deadline.",
    },
    {
        "check_name": "due_present_closed_missing",
        "description": "due_date present and closed_date missing",
        "condition": "due_date IS NOT NULL AND closed_date IS NULL",
        "severity": "review_required",
        "interpretation": "May represent an open or unresolved complaint; eligibility is deferred.",
    },
    {
        "check_name": "closed_status_missing_closed_date",
        "description": "status indicates Closed and closed_date is missing",
        "condition": f"{closed_status_condition} AND closed_date IS NULL",
        "severity": "critical_for_target",
        "interpretation": "Workflow state and outcome timestamp contradict each other.",
    },
    {
        "check_name": "non_closed_status_with_closed_date",
        "description": "observed non-Closed status with closed_date present",
        "condition": f"{non_closed_status_condition} AND closed_date IS NOT NULL",
        "severity": "review_required",
        "interpretation": "May be a workflow transition or a status/timestamp contradiction.",
    },
    {
        "check_name": "latitude_without_longitude",
        "description": "latitude present and longitude missing",
        "condition": "latitude IS NOT NULL AND longitude IS NULL",
        "severity": "warning",
        "interpretation": "Coordinate pair is incomplete.",
    },
    {
        "check_name": "longitude_without_latitude",
        "description": "longitude present and latitude missing",
        "condition": "longitude IS NOT NULL AND latitude IS NULL",
        "severity": "warning",
        "interpretation": "Coordinate pair is incomplete.",
    },
    {
        "check_name": "borough_and_zip_both_missing",
        "description": "borough missing and incident_zip missing",
        "condition": "borough IS NULL AND incident_zip IS NULL",
        "severity": "warning",
        "interpretation": "Two common geographic grouping fields are absent together.",
    },
    {
        "check_name": "borough_missing_coordinates_present",
        "description": "borough missing while both coordinates are present",
        "condition": (
            "borough IS NULL AND latitude IS NOT NULL AND longitude IS NOT NULL"
        ),
        "severity": "review_required",
        "interpretation": "Coordinates may permit later geovalidation without altering raw data.",
    },
]

missing_pattern_query = "SELECT count(*) AS total_rows, " + ", ".join(
    conditional_count_expression(rule["condition"], rule["check_name"])
    for rule in MISSING_PATTERN_RULES
)
missing_pattern_record = fetch_single_aggregate(
    "missing_patterns",
    missing_pattern_query,
)
if missing_pattern_record is None:
    raise RuntimeError("Missing-pattern query did not return a result.")

missing_pattern_summary = pd.DataFrame(
    [
        {
            "check_name": rule["check_name"],
            "description": rule["description"],
            "affected_count": safe_int(
                missing_pattern_record.get(rule["check_name"])
            ),
            "affected_percentage": percentage(
                safe_int(missing_pattern_record.get(rule["check_name"])),
                total_row_count,
            ),
            "severity": rule["severity"],
            "interpretation": rule["interpretation"],
        }
        for rule in MISSING_PATTERN_RULES
    ]
)
missing_pattern_summary.to_csv(
    REPORT_TABLES_DIR / "missing_pattern_summary.csv",
    index=False,
)
missing_pattern_summary


,check_name,description,affected_count,affected_percentage,severity,interpretation
0,closed_and_due_both_missing,closed_date missing and due_date missing,413762,1.887402,critical_for_target,Neither essential input to the initial target concept is present.
1,closed_present_due_missing,closed_date present and due_date missing,21431552,97.761417,critical_for_target,An actual closure exists without the expected deadline.
2,due_present_closed_missing,due_date present and closed_date missing,2031,0.009265,review_required,May represent an open or unresolved complaint; eligibility is deferred.
3,closed_status_missing_closed_date,status indicates Closed and closed_date is missing,53927,0.245992,critical_for_target,Workflow state and outcome timestamp contradict each other.
4,non_closed_status_with_closed_date,observed non-Closed status with closed_date present,78333,0.357321,review_required,May be a workflow transition or a status/timestamp contradiction.
5,latitude_without_longitude,latitude present and longitude missing,0,0.000000,warning,Coordinate pair is incomplete.
6,longitude_without_latitude,longitude present and latitude missing,0,0.000000,warning,Coordinate pair is incomplete.
7,borough_and_zip_both_missing,borough missing and incident_zip missing,38412,0.175219,warning,Two common geographic grouping fields are absent together.
8,borough_missing_coordinates_present,borough missing while both coordinates are present,18,0.000082,review_required,Coordinates may permit later geovalidation without altering raw data.


## 9. Full-dataset duplicate analysis

> **Analysis scope:** Complete live NYC 311 dataset using server-side SoQL
> aggregation with deterministic pagination.

Null identifiers and all duplicate-ID groups are counted over the complete
source. Duplicate conflicts use distinct counts for critical fields. The
exact-row check attempts a grouped query across the configured projection; if
Socrata cannot execute it within the bounded request, the result is explicitly
`not_computable_via_current_api_query`, never zero.


In [9]:
duplicate_unique_key_groups = fetch_paginated_grouped(
    "duplicate_unique_key_groups",
    (
        "SELECT unique_key, count(*) AS record_count "
        "WHERE unique_key IS NOT NULL "
        "GROUP BY unique_key "
        "HAVING count(*) > 1 "
        "ORDER BY unique_key"
    ),
    ["unique_key", "record_count"],
)
if duplicate_unique_key_groups is None:
    raise RuntimeError("Complete duplicate-ID pagination is required.")
if not duplicate_unique_key_groups.empty:
    duplicate_unique_key_groups["record_count"] = pd.to_numeric(
        duplicate_unique_key_groups["record_count"],
        errors="raise",
    ).astype("int64")
duplicate_unique_key_groups.to_csv(
    REPORT_TABLES_DIR / "duplicate_unique_key_groups.csv",
    index=False,
)

duplicate_group_count = int(len(duplicate_unique_key_groups))
rows_in_duplicate_groups = (
    int(duplicate_unique_key_groups["record_count"].sum())
    if duplicate_group_count
    else 0
)
excess_duplicate_rows = rows_in_duplicate_groups - duplicate_group_count
maximum_records_per_unique_key = (
    int(duplicate_unique_key_groups["record_count"].max())
    if duplicate_group_count
    else 1
)
null_unique_key_count = total_row_count - non_null_counts["unique_key"]

CONFLICT_FIELDS = [
    "created_date",
    "closed_date",
    "due_date",
    "agency",
    "complaint_type",
    "status",
    "borough",
]
conflict_output_columns = [
    "duplicate_unique_key",
    "record_count",
    "conflicting_field_count",
    "conflicting_fields",
]

if duplicate_group_count:
    distinct_expressions = ", ".join(
        f"count(distinct {field}) AS {field}_distinct_count"
        for field in CONFLICT_FIELDS
    )
    conflicting_source = fetch_paginated_grouped(
        "conflicting_duplicate_ids",
        (
            "SELECT unique_key AS duplicate_unique_key, "
            "count(*) AS record_count, "
            f"{distinct_expressions} "
            "WHERE unique_key IS NOT NULL "
            "GROUP BY unique_key "
            "HAVING count(*) > 1 "
            "ORDER BY unique_key"
        ),
        [
            "duplicate_unique_key",
            "record_count",
            *[f"{field}_distinct_count" for field in CONFLICT_FIELDS],
        ],
        allow_failure=True,
    )
    if conflicting_source is None:
        conflicting_duplicate_ids = pd.DataFrame(columns=conflict_output_columns)
        conflict_check_status = "not_computable_via_current_api_query"
        conflict_check_notes = "Socrata rejected the grouped distinct-count query."
    else:
        conflict_rows = []
        for row in conflicting_source.to_dict("records"):
            conflicting_fields = [
                field
                for field in CONFLICT_FIELDS
                if (safe_int(row.get(f"{field}_distinct_count")) or 0) > 1
            ]
            if conflicting_fields:
                conflict_rows.append(
                    {
                        "duplicate_unique_key": row["duplicate_unique_key"],
                        "record_count": safe_int(row["record_count"]),
                        "conflicting_field_count": len(conflicting_fields),
                        "conflicting_fields": ", ".join(conflicting_fields),
                    }
                )
        conflicting_duplicate_ids = pd.DataFrame(
            conflict_rows,
            columns=conflict_output_columns,
        )
        conflict_check_status = "computed"
        conflict_check_notes = "All duplicate-ID groups were evaluated."
else:
    conflicting_duplicate_ids = pd.DataFrame(columns=conflict_output_columns)
    conflict_check_status = "computed"
    conflict_check_notes = "No duplicate unique_key groups exist to conflict."

conflicting_duplicate_ids.to_csv(
    REPORT_TABLES_DIR / "conflicting_duplicate_ids.csv",
    index=False,
)

exact_projection = [
    column for column in SELECTED_COLUMNS if column in dataset_columns
]
exact_group_expression = ", ".join(exact_projection)
exact_order_expression = ", ".join(
    column
    for column in exact_projection
    if metadata_by_field[column].get("dataTypeName") != "point"
)
exact_duplicate_groups = fetch_paginated_grouped(
    "exact_duplicate_records",
    (
        f"SELECT {exact_group_expression}, count(*) AS record_count "
        f"GROUP BY {exact_group_expression} "
        "HAVING count(*) > 1 "
        f"ORDER BY {exact_order_expression}"
    ),
    [*exact_projection, "record_count"],
    allow_failure=True,
    timeout_seconds=EXACT_DUPLICATE_TIMEOUT_SECONDS,
)

if exact_duplicate_groups is None:
    exact_duplicate_status = "not_computable_via_current_api_query"
    exact_duplicate_rows = None
    exact_duplicate_excess_rows = None
    exact_failure_reason = next(
        (
            failure["reason"]
            for failure in reversed(QUERY_FAILURES)
            if failure["query_name"].startswith("exact_duplicate_records")
        ),
        "The API did not return a computable result.",
    )
    exact_duplicate_notes = (
        "The bounded full-projection grouped query was rejected or timed out: "
        f"{exact_failure_reason[:500]} "
        "Finalize exact-row duplication after reproducible scoped ingestion."
    )
else:
    if not exact_duplicate_groups.empty:
        exact_duplicate_groups["record_count"] = pd.to_numeric(
            exact_duplicate_groups["record_count"],
            errors="raise",
        ).astype("int64")
    exact_duplicate_status = "computed"
    exact_duplicate_rows = (
        int(exact_duplicate_groups["record_count"].sum())
        if not exact_duplicate_groups.empty
        else 0
    )
    exact_duplicate_excess_rows = (
        exact_duplicate_rows - len(exact_duplicate_groups)
    )
    exact_duplicate_notes = "All exact duplicate projection groups were retrieved."

duplicate_summary = pd.DataFrame(
    [
        {
            "check_name": "null_unique_key",
            "analysis_scope": ANALYSIS_MODE,
            "status": "computed",
            "affected_count": null_unique_key_count,
            "affected_percentage": percentage(
                null_unique_key_count,
                total_row_count,
            ),
            "notes": "Rows where unique_key is null.",
        },
        {
            "check_name": "duplicated_unique_key_groups",
            "analysis_scope": ANALYSIS_MODE,
            "status": "computed",
            "affected_count": duplicate_group_count,
            "affected_percentage": None,
            "notes": "Number of identifiers appearing more than once.",
        },
        {
            "check_name": "rows_in_duplicate_id_groups",
            "analysis_scope": ANALYSIS_MODE,
            "status": "computed",
            "affected_count": rows_in_duplicate_groups,
            "affected_percentage": percentage(
                rows_in_duplicate_groups,
                total_row_count,
            ),
            "notes": "Sum of record_count across duplicate-ID groups.",
        },
        {
            "check_name": "excess_duplicate_id_rows",
            "analysis_scope": ANALYSIS_MODE,
            "status": "computed",
            "affected_count": excess_duplicate_rows,
            "affected_percentage": percentage(
                excess_duplicate_rows,
                total_row_count,
            ),
            "notes": "Sum of record_count - 1 across duplicate-ID groups.",
        },
        {
            "check_name": "maximum_records_for_one_unique_key",
            "analysis_scope": ANALYSIS_MODE,
            "status": "computed",
            "affected_count": maximum_records_per_unique_key,
            "affected_percentage": None,
            "notes": "Maximum source rows associated with one non-null identifier.",
        },
        {
            "check_name": "conflicting_duplicate_ids",
            "analysis_scope": ANALYSIS_MODE,
            "status": conflict_check_status,
            "affected_count": (
                len(conflicting_duplicate_ids)
                if conflict_check_status == "computed"
                else None
            ),
            "affected_percentage": None,
            "notes": conflict_check_notes,
        },
        {
            "check_name": "exact_duplicate_rows",
            "analysis_scope": ANALYSIS_MODE,
            "status": exact_duplicate_status,
            "affected_count": exact_duplicate_rows,
            "affected_percentage": percentage(
                exact_duplicate_rows,
                total_row_count,
            ),
            "notes": exact_duplicate_notes,
        },
        {
            "check_name": "excess_exact_duplicate_rows",
            "analysis_scope": ANALYSIS_MODE,
            "status": exact_duplicate_status,
            "affected_count": exact_duplicate_excess_rows,
            "affected_percentage": percentage(
                exact_duplicate_excess_rows,
                total_row_count,
            ),
            "notes": exact_duplicate_notes,
        },
    ]
)
duplicate_summary.to_csv(
    REPORT_TABLES_DIR / "duplicate_summary.csv",
    index=False,
)

display(duplicate_summary)
display(duplicate_unique_key_groups.head(20))
display(conflicting_duplicate_ids.head(20))


,check_name,analysis_scope,status,affected_count,affected_percentage,notes
0,null_unique_key,full-dataset server-side aggregation,computed,0,0.0,Rows where unique_key is null.
1,duplicated_unique_key_groups,full-dataset server-side aggregation,computed,0,NaN,Number of identifiers appearing more than once.
2,rows_in_duplicate_id_groups,full-dataset server-side aggregation,computed,0,0.0,Sum of record_count across duplicate-ID groups.
3,excess_duplicate_id_rows,full-dataset server-side aggregation,computed,0,0.0,Sum of record_count - 1 across duplicate-ID groups.
4,maximum_records_for_one_unique_key,full-dataset server-side aggregation,computed,1,NaN,Maximum source rows associated with one non-null identifier.
5,conflicting_duplicate_ids,full-dataset server-side aggregation,computed,0,NaN,No duplicate unique_key groups exist to conflict.
6,exact_duplicate_rows,full-dataset server-side aggregation,computed,0,0.0,All exact duplicate projection groups were retrieved.
7,excess_exact_duplicate_rows,full-dataset server-side aggregation,computed,0,0.0,All exact duplicate projection groups were retrieved.


,unique_key,record_count


,duplicate_unique_key,record_count,conflicting_field_count,conflicting_fields


## 10. Full-dataset timestamp field quality

> **Analysis scope:** Complete live NYC 311 dataset using server-side SoQL
> aggregation.

The source metadata establishes datetime types. Counts and ranges are computed
over the complete dataset. Source-level format validity is enforced by the
API's typed datetime schema; downstream pandas parsing will still be validated
after ingestion. No sample-based parse rate is used.


In [10]:
timestamp_expressions = ["count(*) AS total_rows"]
for column_name in TIMESTAMP_COLUMNS:
    timestamp_expressions.extend(
        [
            f"count({column_name}) AS {column_name}_non_null",
            f"min({column_name}) AS {column_name}_minimum",
            f"max({column_name}) AS {column_name}_maximum",
        ]
    )

timestamp_field_record = fetch_single_aggregate(
    "timestamp_field_summary",
    "SELECT " + ", ".join(timestamp_expressions),
)
if timestamp_field_record is None:
    raise RuntimeError("Timestamp field summary query did not return a result.")

timestamp_field_rows = []
for column_name in TIMESTAMP_COLUMNS:
    source_type = str(
        metadata_by_field.get(column_name, {}).get("dataTypeName", "unavailable")
    )
    non_null_count = safe_int(
        timestamp_field_record.get(f"{column_name}_non_null")
    )
    missing_count = (
        total_row_count - non_null_count
        if non_null_count is not None
        else None
    )
    timestamp_field_rows.append(
        {
            "column_name": column_name,
            "source_data_type": source_type,
            "total_rows": total_row_count,
            "non_null_count": non_null_count,
            "missing_count": missing_count,
            "missing_percentage": percentage(missing_count, total_row_count),
            "minimum_timestamp": timestamp_field_record.get(
                f"{column_name}_minimum"
            ),
            "maximum_timestamp": timestamp_field_record.get(
                f"{column_name}_maximum"
            ),
            "source_type_valid": source_type == "calendar_date",
            "format_validity_note": (
                "Source-level format validity is enforced by the API's typed "
                "datetime schema; validate pandas parsing after ingestion."
            ),
        }
    )

timestamp_field_summary = pd.DataFrame(timestamp_field_rows)
timestamp_field_summary.to_csv(
    REPORT_TABLES_DIR / "timestamp_field_summary.csv",
    index=False,
)
timestamp_field_summary


,column_name,source_data_type,total_rows,non_null_count,missing_count,missing_percentage,minimum_timestamp,maximum_timestamp,source_type_valid,format_validity_note
0,created_date,calendar_date,21922301,21922301,0,0.000000,2020-01-01T00:00:00.000,2026-07-24T01:50:58.000,True,Source-level format validity is enforced by the API's typed datetime schema; validate pandas parsing after ingestion.
1,closed_date,calendar_date,21922301,21506508,415793,1.896667,1899-12-31T19:00:00.000,2033-03-01T00:00:00.000,True,Source-level format validity is enforced by the API's typed datetime schema; validate pandas parsing after ingestion.
2,due_date,calendar_date,21922301,76987,21845314,99.648819,2018-09-29T09:26:39.000,2026-08-22T20:02:47.000,True,Source-level format validity is enforced by the API's typed datetime schema; validate pandas parsing after ingestion.
3,resolution_action_updated_date,calendar_date,21922301,21742539,179762,0.819996,2019-07-18T00:00:00.000,2026-12-14T11:50:00.000,True,Source-level format validity is enforced by the API's typed datetime schema; validate pandas parsing after ingestion.


## 11. Full-dataset timestamp consistency

> **Analysis scope:** Complete live NYC 311 dataset using server-side SoQL
> aggregation.

Chronology checks count every violating row. A future `due_date` is
informational rather than invalid by itself because an open complaint can
legitimately have a future deadline.


In [11]:
analysis_timestamp_literal = analysis_timestamp_utc.strftime(
    "%Y-%m-%dT%H:%M:%S"
)
TIMESTAMP_CONSISTENCY_RULES = [
    {
        "check_name": "closed_before_created",
        "business_rule": "closed_date must not precede created_date",
        "condition": (
            "closed_date IS NOT NULL AND created_date IS NOT NULL "
            "AND closed_date < created_date"
        ),
        "severity": "critical_for_target",
        "interpretation": "Negative complaint lifetime undermines outcome chronology.",
    },
    {
        "check_name": "due_before_created",
        "business_rule": "due_date should not precede created_date",
        "condition": (
            "due_date IS NOT NULL AND created_date IS NOT NULL "
            "AND due_date < created_date"
        ),
        "severity": "critical_for_target",
        "interpretation": "Expected deadline predates the prediction moment.",
    },
    {
        "check_name": "resolution_update_before_created",
        "business_rule": (
            "resolution_action_updated_date should not precede created_date"
        ),
        "condition": (
            "resolution_action_updated_date IS NOT NULL "
            "AND created_date IS NOT NULL "
            "AND resolution_action_updated_date < created_date"
        ),
        "severity": "warning",
        "interpretation": "Recorded workflow update predates complaint creation.",
    },
    {
        "check_name": "created_after_analysis_time",
        "business_rule": "created_date must not be later than extraction time",
        "condition": f"created_date > '{analysis_timestamp_literal}'",
        "severity": "warning",
        "interpretation": "Creation timestamp is in the future relative to extraction.",
    },
    {
        "check_name": "closed_after_analysis_time",
        "business_rule": "closed_date must not be later than extraction time",
        "condition": f"closed_date > '{analysis_timestamp_literal}'",
        "severity": "warning",
        "interpretation": "Outcome timestamp is in the future relative to extraction.",
    },
    {
        "check_name": "resolution_update_after_analysis_time",
        "business_rule": (
            "resolution_action_updated_date must not be later than extraction time"
        ),
        "condition": (
            f"resolution_action_updated_date > '{analysis_timestamp_literal}'"
        ),
        "severity": "warning",
        "interpretation": "Workflow update is in the future relative to extraction.",
    },
    {
        "check_name": "due_after_analysis_time",
        "business_rule": (
            "Future due dates are permitted and should be reviewed with status"
        ),
        "condition": f"due_date > '{analysis_timestamp_literal}'",
        "severity": "informational",
        "interpretation": "May be a legitimate future SLA deadline for an open complaint.",
    },
]

timestamp_quality_record = fetch_single_aggregate(
    "timestamp_consistency",
    "SELECT count(*) AS total_rows, "
    + ", ".join(
        conditional_count_expression(rule["condition"], rule["check_name"])
        for rule in TIMESTAMP_CONSISTENCY_RULES
    ),
)
if timestamp_quality_record is None:
    raise RuntimeError("Timestamp consistency query did not return a result.")

timestamp_quality_summary = pd.DataFrame(
    [
        {
            "check_name": rule["check_name"],
            "business_rule": rule["business_rule"],
            "affected_count": safe_int(
                timestamp_quality_record.get(rule["check_name"])
            ),
            "affected_percentage": percentage(
                safe_int(timestamp_quality_record.get(rule["check_name"])),
                total_row_count,
            ),
            "severity": rule["severity"],
            "interpretation": rule["interpretation"],
        }
        for rule in TIMESTAMP_CONSISTENCY_RULES
    ]
)
timestamp_quality_summary.to_csv(
    REPORT_TABLES_DIR / "timestamp_quality_summary.csv",
    index=False,
)
timestamp_quality_summary


,check_name,business_rule,affected_count,affected_percentage,severity,interpretation
0,closed_before_created,closed_date must not precede created_date,46492,0.212076,critical_for_target,Negative complaint lifetime undermines outcome chronology.
1,due_before_created,due_date should not precede created_date,16,0.000073,critical_for_target,Expected deadline predates the prediction moment.
2,resolution_update_before_created,resolution_action_updated_date should not precede created_date,498960,2.276038,warning,Recorded workflow update predates complaint creation.
3,created_after_analysis_time,created_date must not be later than extraction time,0,0.000000,warning,Creation timestamp is in the future relative to extraction.
4,closed_after_analysis_time,closed_date must not be later than extraction time,2,0.000009,warning,Outcome timestamp is in the future relative to extraction.
5,resolution_update_after_analysis_time,resolution_action_updated_date must not be later than extraction time,1,0.000005,warning,Workflow update is in the future relative to extraction.
6,due_after_analysis_time,Future due dates are permitted and should be reviewed with status,1116,0.005091,informational,May be a legitimate future SLA deadline for an open complaint.


## 12. Complete status/timestamp consistency

> **Analysis scope:** Complete live NYC 311 dataset using server-side SoQL
> aggregation.

These checks use the observed status categories defined above. They assess
eligibility and contradictions without treating final status as a feature or
making final target-eligibility decisions.


In [12]:
active_status_condition = (
    f"status IN {soql_string_list(OPEN_OR_ACTIVE_STATUSES)}"
    if OPEN_OR_ACTIVE_STATUSES
    else "false"
)
special_status_condition = (
    f"status IN {soql_string_list(EXCLUDED_OR_SPECIAL_STATUSES)}"
    if EXCLUDED_OR_SPECIAL_STATUSES
    else "false"
)

STATUS_TIMESTAMP_RULES = [
    {
        "check_name": "closed_status_missing_closed_date",
        "business_rule": "Closed status should have a closed_date",
        "condition": f"{closed_status_condition} AND closed_date IS NULL",
        "severity": "critical_for_target",
        "interpretation": "Closure status lacks its outcome timestamp.",
    },
    {
        "check_name": "active_status_with_closed_date",
        "business_rule": "Open or active status is expected to lack final closure",
        "condition": f"{active_status_condition} AND closed_date IS NOT NULL",
        "severity": "review_required",
        "interpretation": "May reflect a reopened complaint or a workflow inconsistency.",
    },
    {
        "check_name": "closed_status_closed_before_created",
        "business_rule": "Closed complaints must not close before creation",
        "condition": (
            f"{closed_status_condition} AND closed_date IS NOT NULL "
            "AND created_date IS NOT NULL AND closed_date < created_date"
        ),
        "severity": "critical_for_target",
        "interpretation": "Closed workflow state has impossible outcome chronology.",
    },
    {
        "check_name": "special_status_with_closed_date",
        "business_rule": "Special or excluded statuses require explicit eligibility review",
        "condition": f"{special_status_condition} AND closed_date IS NOT NULL",
        "severity": "review_required",
        "interpretation": "Closure information exists for a special-status complaint.",
    },
]

status_timestamp_record = fetch_single_aggregate(
    "status_timestamp_consistency",
    "SELECT count(*) AS total_rows, "
    + ", ".join(
        conditional_count_expression(rule["condition"], rule["check_name"])
        for rule in STATUS_TIMESTAMP_RULES
    ),
)
if status_timestamp_record is None:
    raise RuntimeError("Status/timestamp consistency query did not return a result.")

status_timestamp_consistency = pd.DataFrame(
    [
        {
            "check_name": rule["check_name"],
            "business_rule": rule["business_rule"],
            "affected_count": safe_int(
                status_timestamp_record.get(rule["check_name"])
            ),
            "affected_percentage": percentage(
                safe_int(status_timestamp_record.get(rule["check_name"])),
                total_row_count,
            ),
            "severity": rule["severity"],
            "interpretation": rule["interpretation"],
        }
        for rule in STATUS_TIMESTAMP_RULES
    ]
)
status_timestamp_consistency.to_csv(
    REPORT_TABLES_DIR / "status_timestamp_consistency.csv",
    index=False,
)
status_timestamp_consistency


,check_name,business_rule,affected_count,affected_percentage,severity,interpretation
0,closed_status_missing_closed_date,Closed status should have a closed_date,53927,0.245992,critical_for_target,Closure status lacks its outcome timestamp.
1,active_status_with_closed_date,Open or active status is expected to lack final closure,75646,0.345064,review_required,May reflect a reopened complaint or a workflow inconsistency.
2,closed_status_closed_before_created,Closed complaints must not close before creation,1012,0.004616,critical_for_target,Closed workflow state has impossible outcome chronology.
3,special_status_with_closed_date,Special or excluded statuses require explicit eligibility review,2687,0.012257,review_required,Closure information exists for a special-status complaint.


## 13. Full-dataset categorical quality

> **Analysis scope:** Complete live NYC 311 dataset using complete server-side
> grouped SoQL aggregations.

Each raw categorical distribution is fully paginated. Empty strings,
whitespace variants, and placeholder labels are calculated from those complete
group counts. Case collisions use the supported server-side `lower(...)`
expression while retaining raw variants. Placeholder values are reported for
review, not automatically rejected.


In [13]:
categorical_quality_rows: list[dict[str, object]] = []
case_collision_frames: list[pd.DataFrame] = []
raw_categorical_groups: dict[str, pd.DataFrame] = {}


def append_categorical_check(
    column_name: str,
    check_name: str,
    status: str,
    affected_count: int | None,
    interpretation: str,
) -> None:
    """Append one categorical result with explicit computability."""
    categorical_quality_rows.append(
        {
            "column_name": column_name,
            "check_name": check_name,
            "analysis_scope": ANALYSIS_MODE,
            "status": status,
            "affected_count": affected_count,
            "affected_percentage": percentage(
                affected_count,
                total_row_count,
            ),
            "interpretation": interpretation,
        }
    )


for column_name in CATEGORICAL_QUALITY_COLUMNS:
    raw_groups = fetch_paginated_grouped(
        f"categorical_raw_{column_name}",
        (
            f"SELECT {column_name} AS raw_value, count(*) AS record_count "
            f"GROUP BY {column_name} ORDER BY {column_name}"
        ),
        ["raw_value", "record_count"],
        allow_failure=True,
    )
    if raw_groups is None:
        for check_name in [
            "null_values",
            "empty_strings",
            "leading_or_trailing_whitespace",
            "placeholder_values",
        ]:
            append_categorical_check(
                column_name,
                check_name,
                "not_computable",
                None,
                "The complete raw grouped query failed; no sample substitute was used.",
            )
    else:
        raw_groups["record_count"] = pd.to_numeric(
            raw_groups["record_count"],
            errors="raise",
        ).astype("int64")
        raw_categorical_groups[column_name] = raw_groups
        null_count = int(
            raw_groups.loc[raw_groups["raw_value"].isna(), "record_count"].sum()
        )
        non_null_groups = raw_groups.loc[raw_groups["raw_value"].notna()].copy()
        non_null_groups["raw_text"] = non_null_groups["raw_value"].astype(str)
        empty_count = int(
            non_null_groups.loc[
                non_null_groups["raw_text"] == "",
                "record_count",
            ].sum()
        )
        whitespace_count = int(
            non_null_groups.loc[
                non_null_groups["raw_text"]
                != non_null_groups["raw_text"].str.strip(),
                "record_count",
            ].sum()
        )
        placeholder_count = int(
            non_null_groups.loc[
                non_null_groups["raw_text"].str.strip().str.casefold().isin(
                    PLACEHOLDER_VALUES
                ),
                "record_count",
            ].sum()
        )
        append_categorical_check(
            column_name,
            "null_values",
            "computed",
            null_count,
            "Null source values in the complete grouped distribution.",
        )
        append_categorical_check(
            column_name,
            "empty_strings",
            "computed",
            empty_count,
            "Present-but-empty strings; not the same as API nulls.",
        )
        append_categorical_check(
            column_name,
            "leading_or_trailing_whitespace",
            "computed",
            whitespace_count,
            "Raw category labels with surrounding whitespace.",
        )
        append_categorical_check(
            column_name,
            "placeholder_values",
            "computed",
            placeholder_count,
            "Configured placeholder-like labels reported for review, not rejection.",
        )

    normalized_groups = fetch_paginated_grouped(
        f"categorical_case_{column_name}",
        (
            f"SELECT lower({column_name}) AS normalized_value, "
            f"{column_name} AS raw_value, count(*) AS record_count "
            f"WHERE {column_name} IS NOT NULL "
            f"GROUP BY lower({column_name}), {column_name} "
            f"ORDER BY lower({column_name}), {column_name}"
        ),
        ["normalized_value", "raw_value", "record_count"],
        allow_failure=True,
    )
    if normalized_groups is None:
        append_categorical_check(
            column_name,
            "case_insensitive_collisions",
            "not_computable",
            None,
            "Socrata rejected lower(...) grouping; no sample substitute was used.",
        )
        continue

    normalized_groups["record_count"] = pd.to_numeric(
        normalized_groups["record_count"],
        errors="raise",
    ).astype("int64")
    variant_counts = (
        normalized_groups.groupby("normalized_value", dropna=False)["raw_value"]
        .nunique(dropna=True)
        .rename("raw_variant_count")
    )
    collision_keys = variant_counts.loc[variant_counts > 1].index
    collision_source = normalized_groups.loc[
        normalized_groups["normalized_value"].isin(collision_keys)
    ]
    collision_rows = []
    for normalized_value, group in collision_source.groupby(
        "normalized_value",
        dropna=False,
        sort=True,
    ):
        collision_rows.append(
            {
                "column_name": column_name,
                "normalized_value": normalized_value,
                "raw_variant_count": int(group["raw_value"].nunique()),
                "raw_variants": " | ".join(
                    sorted(group["raw_value"].astype(str).unique())
                ),
                "affected_count": int(group["record_count"].sum()),
                "analysis_scope": ANALYSIS_MODE,
            }
        )
    collision_frame = pd.DataFrame(
        collision_rows,
        columns=[
            "column_name",
            "normalized_value",
            "raw_variant_count",
            "raw_variants",
            "affected_count",
            "analysis_scope",
        ],
    )
    case_collision_frames.append(collision_frame)
    append_categorical_check(
        column_name,
        "case_insensitive_collisions",
        "computed",
        (
            int(collision_frame["affected_count"].sum())
            if not collision_frame.empty
            else 0
        ),
        "Rows in lowercased category groups containing multiple raw variants.",
    )

categorical_quality_summary = pd.DataFrame(categorical_quality_rows)
categorical_quality_summary.to_csv(
    REPORT_TABLES_DIR / "categorical_quality_summary.csv",
    index=False,
)

categorical_case_collisions = (
    pd.concat(case_collision_frames, ignore_index=True)
    if case_collision_frames
    else pd.DataFrame(
        columns=[
            "column_name",
            "normalized_value",
            "raw_variant_count",
            "raw_variants",
            "affected_count",
            "analysis_scope",
        ]
    )
)
categorical_case_collisions.to_csv(
    REPORT_TABLES_DIR / "categorical_case_collisions.csv",
    index=False,
)

agency_mapping_groups = fetch_paginated_grouped(
    "agency_name_mapping",
    (
        "SELECT agency, agency_name, count(*) AS record_count "
        "WHERE agency IS NOT NULL "
        "GROUP BY agency, agency_name "
        "ORDER BY agency, agency_name"
    ),
    ["agency", "agency_name", "record_count"],
    allow_failure=True,
)
if agency_mapping_groups is None:
    agency_name_mapping_issues = pd.DataFrame(
        [
            {
                "agency": None,
                "agency_name_variant_count": None,
                "agency_names": None,
                "affected_count": None,
                "status": "not_computable",
                "reason": "Socrata rejected the complete agency/name grouped query.",
            }
        ]
    )
else:
    agency_mapping_groups["record_count"] = pd.to_numeric(
        agency_mapping_groups["record_count"],
        errors="raise",
    ).astype("int64")
    mapping_rows = []
    for agency, group in agency_mapping_groups.groupby("agency", sort=True):
        names = sorted(group["agency_name"].dropna().astype(str).unique())
        if len(names) > 1:
            mapping_rows.append(
                {
                    "agency": agency,
                    "agency_name_variant_count": len(names),
                    "agency_names": " | ".join(names),
                    "affected_count": int(group["record_count"].sum()),
                    "status": "computed",
                    "reason": "One agency code maps to multiple non-null source labels.",
                }
            )
    agency_name_mapping_issues = pd.DataFrame(
        mapping_rows,
        columns=[
            "agency",
            "agency_name_variant_count",
            "agency_names",
            "affected_count",
            "status",
            "reason",
        ],
    )

agency_name_mapping_issues.to_csv(
    REPORT_TABLES_DIR / "agency_name_mapping_issues.csv",
    index=False,
)

display(categorical_quality_summary)
display(agency_name_mapping_issues.head(20))
display(categorical_case_collisions.head(20))


,column_name,check_name,analysis_scope,status,affected_count,affected_percentage,interpretation
0,agency,null_values,full-dataset server-side aggregation,computed,0,0.000000,Null source values in the complete grouped distribution.
1,agency,empty_strings,full-dataset server-side aggregation,computed,0,0.000000,Present-but-empty strings; not the same as API nulls.
2,agency,leading_or_trailing_whitespace,full-dataset server-side aggregation,computed,0,0.000000,Raw category labels with surrounding whitespace.
3,agency,placeholder_values,full-dataset server-side aggregation,computed,0,0.000000,"Configured placeholder-like labels reported for review, not rejection."
4,agency,case_insensitive_collisions,full-dataset server-side aggregation,computed,0,0.000000,Rows in lowercased category groups containing multiple raw variants.
5,agency_name,null_values,full-dataset server-side aggregation,computed,0,0.000000,Null source values in the complete grouped distribution.
6,agency_name,empty_strings,full-dataset server-side aggregation,computed,0,0.000000,Present-but-empty strings; not the same as API nulls.
7,agency_name,leading_or_trailing_whitespace,full-dataset server-side aggregation,computed,0,0.000000,Raw category labels with surrounding whitespace.
8,agency_name,placeholder_values,full-dataset server-side aggregation,computed,0,0.000000,"Configured placeholder-like labels reported for review, not rejection."
9,agency_name,case_insensitive_collisions,full-dataset server-side aggregation,computed,0,0.000000,Rows in lowercased category groups containing multiple raw variants.


,agency,agency_name_variant_count,agency_names,affected_count,status,reason
0,DHS,2,Department of Homeless Services | Operations Unit - Department of Homeless Services,288706,computed,One agency code maps to multiple non-null source labels.


,column_name,normalized_value,raw_variant_count,raw_variants,affected_count,analysis_scope
0,complaint_type,appliance,2,APPLIANCE | Appliance,132883,full-dataset server-side aggregation
1,complaint_type,asbestos,2,ASBESTOS | Asbestos,9302,full-dataset server-side aggregation
2,complaint_type,door/window,2,DOOR/WINDOW | Door/Window,270476,full-dataset server-side aggregation
3,complaint_type,electric,2,ELECTRIC | Electric,175895,full-dataset server-side aggregation
4,complaint_type,elevator,2,ELEVATOR | Elevator,129980,full-dataset server-side aggregation
5,complaint_type,flooring/stairs,2,FLOORING/STAIRS | Flooring/Stairs,158758,full-dataset server-side aggregation
6,complaint_type,general,2,GENERAL | General,191728,full-dataset server-side aggregation
7,complaint_type,heat/hot water,2,HEAT/HOT WATER | Heat/Hot Water,1646961,full-dataset server-side aggregation
8,complaint_type,mold,2,MOLD | Mold,2075,full-dataset server-side aggregation
9,complaint_type,outside building,2,OUTSIDE BUILDING | Outside Building,6158,full-dataset server-side aggregation


## 14. Full-dataset geographic quality

> **Analysis scope:** Complete live NYC 311 dataset using server-side SoQL
> aggregation.

Global coordinate limits are structural checks. The approximate NYC bounding
box is configurable and informational only: records outside it are not
automatically declared invalid. ZIP format is assessed exactly from a complete,
paginated server-side grouped distribution without modifying source values.


In [14]:
minimum_latitude = APPROXIMATE_NYC_BOUNDS["minimum_latitude"]
maximum_latitude = APPROXIMATE_NYC_BOUNDS["maximum_latitude"]
minimum_longitude = APPROXIMATE_NYC_BOUNDS["minimum_longitude"]
maximum_longitude = APPROXIMATE_NYC_BOUNDS["maximum_longitude"]

GEOGRAPHIC_RULES = [
    {
        "check_name": "latitude_outside_global_range",
        "condition": "latitude IS NOT NULL AND (latitude < -90 OR latitude > 90)",
        "severity": "warning",
        "interpretation": "Latitude is outside the mathematically valid global range.",
    },
    {
        "check_name": "longitude_outside_global_range",
        "condition": (
            "longitude IS NOT NULL AND (longitude < -180 OR longitude > 180)"
        ),
        "severity": "warning",
        "interpretation": "Longitude is outside the mathematically valid global range.",
    },
    {
        "check_name": "latitude_without_longitude",
        "condition": "latitude IS NOT NULL AND longitude IS NULL",
        "severity": "warning",
        "interpretation": "Coordinate pair is incomplete.",
    },
    {
        "check_name": "longitude_without_latitude",
        "condition": "longitude IS NOT NULL AND latitude IS NULL",
        "severity": "warning",
        "interpretation": "Coordinate pair is incomplete.",
    },
    {
        "check_name": "both_coordinates_missing",
        "condition": "latitude IS NULL AND longitude IS NULL",
        "severity": "informational",
        "interpretation": "No coordinate pair is available for geographic modelling.",
    },
    {
        "check_name": "borough_missing_coordinates_present",
        "condition": (
            "borough IS NULL AND latitude IS NOT NULL AND longitude IS NOT NULL"
        ),
        "severity": "review_required",
        "interpretation": "Coordinates exist while borough grouping is absent.",
    },
    {
        "check_name": "coordinates_outside_approximate_nyc_box",
        "condition": (
            "latitude IS NOT NULL AND longitude IS NOT NULL AND "
            f"(latitude < {minimum_latitude} OR latitude > {maximum_latitude} "
            f"OR longitude < {minimum_longitude} "
            f"OR longitude > {maximum_longitude})"
        ),
        "severity": "informational",
        "interpretation": (
            "Outside a broad approximate NYC box; inspect context before judging validity."
        ),
    },
]

geographic_record = fetch_single_aggregate(
    "geographic_quality",
    "SELECT count(*) AS total_rows, "
    + ", ".join(
        conditional_count_expression(rule["condition"], rule["check_name"])
        for rule in GEOGRAPHIC_RULES
    ),
)
if geographic_record is None:
    raise RuntimeError("Geographic quality query did not return a result.")

geographic_quality_summary = pd.DataFrame(
    [
        {
            "check_name": rule["check_name"],
            "affected_count": safe_int(
                geographic_record.get(rule["check_name"])
            ),
            "affected_percentage": percentage(
                safe_int(geographic_record.get(rule["check_name"])),
                total_row_count,
            ),
            "severity": rule["severity"],
            "interpretation": rule["interpretation"],
        }
        for rule in GEOGRAPHIC_RULES
    ]
)

incident_zip_groups = fetch_paginated_grouped(
    "incident_zip_format_groups",
    (
        "SELECT incident_zip, count(*) AS record_count "
        "WHERE incident_zip IS NOT NULL "
        "GROUP BY incident_zip ORDER BY incident_zip"
    ),
    ["incident_zip", "record_count"],
    allow_failure=True,
)
if incident_zip_groups is None:
    incident_zip_result = {
        "check_name": "incident_zip_malformed",
        "affected_count": None,
        "affected_percentage": None,
        "severity": "not_computable",
        "interpretation": (
            "The complete grouped ZIP query failed; no sample substitute was used."
        ),
    }
else:
    incident_zip_groups["record_count"] = pd.to_numeric(
        incident_zip_groups["record_count"],
        errors="raise",
    ).astype("int64")
    malformed_zip_count = int(
        incident_zip_groups.loc[
            ~incident_zip_groups["incident_zip"].astype(str).str.fullmatch(
                r"[0-9]{5}"
            ),
            "record_count",
        ].sum()
    )
    incident_zip_result = {
        "check_name": "incident_zip_malformed",
        "affected_count": malformed_zip_count,
        "affected_percentage": percentage(
            malformed_zip_count,
            total_row_count,
        ),
        "severity": "review_required",
        "interpretation": "ZIP is not exactly five ASCII digits.",
    }

geographic_quality_summary = pd.concat(
    [
        geographic_quality_summary,
        pd.DataFrame([incident_zip_result]),
    ],
    ignore_index=True,
)
geographic_quality_summary.to_csv(
    REPORT_TABLES_DIR / "geographic_quality_summary.csv",
    index=False,
)
geographic_quality_summary


,check_name,affected_count,affected_percentage,severity,interpretation
0,latitude_outside_global_range,0,0.000000,warning,Latitude is outside the mathematically valid global range.
1,longitude_outside_global_range,0,0.000000,warning,Longitude is outside the mathematically valid global range.
2,latitude_without_longitude,0,0.000000,warning,Coordinate pair is incomplete.
3,longitude_without_latitude,0,0.000000,warning,Coordinate pair is incomplete.
4,both_coordinates_missing,410402,1.872075,informational,No coordinate pair is available for geographic modelling.
5,borough_missing_coordinates_present,18,0.000082,review_required,Coordinates exist while borough grouping is absent.
6,coordinates_outside_approximate_nyc_box,1,0.000005,informational,Outside a broad approximate NYC box; inspect context before judging validity.
7,incident_zip_malformed,24,0.000109,review_required,ZIP is not exactly five ASCII digits.


## 15. Integrated severity-based issue register

> **Analysis scope:** Complete live NYC 311 dataset using server-side SoQL
> aggregation.

The register integrates schema, missingness, missing patterns, duplicates,
timestamps, status consistency, categorical quality, geography, and API
limitations. Failed checks retain null counts and `not_computable` severity;
zero is reserved for successfully computed zero findings.


In [15]:
issue_rows: list[dict[str, object]] = []


def add_issue(
    severity: str,
    category: str,
    check_name: str,
    column_name: str,
    affected_count: int | None,
    affected_percentage: float | None,
    issue: str,
    impact: str,
    recommended_next_action: str,
) -> None:
    """Append one normalized issue-register record."""
    issue_rows.append(
        {
            "severity": severity,
            "category": category,
            "check_name": check_name,
            "column_name": column_name,
            "affected_count": affected_count,
            "affected_percentage": affected_percentage,
            "analysis_scope": ANALYSIS_MODE,
            "issue": issue,
            "impact": impact,
            "recommended_next_action": recommended_next_action,
        }
    )


for row in column_requirement_validation.to_dict("records"):
    if not row["is_present"]:
        add_issue(
            row["severity_if_missing"],
            "schema_validation",
            "required_column_missing",
            row["column_name"],
            None,
            None,
            f"Configured {row['column_group']} field is absent from API metadata.",
            row["impact_if_missing"],
            "Confirm source schema or revise the documented project contract.",
        )

for row in missing_values_summary.to_dict("records"):
    missing_count = safe_int(row["missing_count"])
    if missing_count is None:
        continue
    if missing_count > 0:
        if row["column_name"] in ESSENTIAL_TARGET_COLUMNS:
            severity = "critical_for_target"
        elif row["column_name"] in CORE_REQUIRED_COLUMNS:
            severity = "warning"
        elif row["missingness_level"] in {"high", "complete"}:
            severity = "review_required"
        else:
            severity = "informational"
        add_issue(
            severity,
            "missingness",
            "column_missing_values",
            row["column_name"],
            missing_count,
            safe_float(row["missing_percentage"]),
            f"{row['column_name']} has {row['missingness_level']} missingness.",
            row["quality_implication"],
            "Assess operational meaning and scoped coverage before any treatment.",
        )

for row in missing_pattern_summary.to_dict("records"):
    affected_count = safe_int(row["affected_count"])
    if affected_count and affected_count > 0:
        add_issue(
            row["severity"],
            "missing_patterns",
            row["check_name"],
            "",
            affected_count,
            safe_float(row["affected_percentage"]),
            row["description"],
            row["interpretation"],
            "Define scoped eligibility or validation handling in Day 4.",
        )

for check_name in [
    "null_unique_key",
    "duplicated_unique_key_groups",
    "excess_duplicate_id_rows",
    "conflicting_duplicate_ids",
    "exact_duplicate_rows",
]:
    row = duplicate_summary.loc[
        duplicate_summary["check_name"] == check_name
    ].iloc[0]
    if row["status"] != "computed":
        add_issue(
            "not_computable",
            "duplicates",
            check_name,
            "unique_key" if "unique_key" in check_name else "",
            None,
            None,
            f"{check_name} could not be completed through the current API query.",
            row["notes"],
            "Finalize this check after reproducible scoped ingestion.",
        )
    else:
        affected_count = safe_int(row["affected_count"])
        if affected_count and affected_count > 0:
            add_issue(
                "critical" if check_name != "exact_duplicate_rows" else "warning",
                "duplicates",
                check_name,
                "unique_key" if "unique_key" in check_name else "",
                affected_count,
                safe_float(row["affected_percentage"]),
                f"{check_name} detected records requiring integrity review.",
                row["notes"],
                "Investigate source identity semantics before complaint-level ingestion.",
            )

for frame, category in [
    (timestamp_quality_summary, "timestamp_consistency"),
    (status_timestamp_consistency, "status_timestamp_consistency"),
]:
    for row in frame.to_dict("records"):
        affected_count = safe_int(row["affected_count"])
        if affected_count and affected_count > 0:
            add_issue(
                row["severity"],
                category,
                row["check_name"],
                "",
                affected_count,
                safe_float(row["affected_percentage"]),
                row["business_rule"],
                row["interpretation"],
                "Review scope and eligibility implications in Day 4.",
            )

for row in categorical_quality_summary.to_dict("records"):
    if row["status"] != "computed":
        add_issue(
            "not_computable",
            "categorical_quality",
            row["check_name"],
            row["column_name"],
            None,
            None,
            f"Categorical check could not be computed for {row['column_name']}.",
            row["interpretation"],
            "Re-run through scoped ingestion without using a sample estimate.",
        )
        continue
    affected_count = safe_int(row["affected_count"])
    if affected_count and affected_count > 0:
        add_issue(
            "review_required",
            "categorical_quality",
            row["check_name"],
            row["column_name"],
            affected_count,
            safe_float(row["affected_percentage"]),
            f"{row['check_name']} detected for {row['column_name']}.",
            row["interpretation"],
            "Review category semantics before defining any normalization rule.",
        )

if not agency_name_mapping_issues.empty:
    computable_mapping_rows = agency_name_mapping_issues.loc[
        agency_name_mapping_issues["status"] == "computed"
    ]
    if computable_mapping_rows.empty and (
        agency_name_mapping_issues["status"] == "not_computable"
    ).any():
        add_issue(
            "not_computable",
            "categorical_quality",
            "agency_name_mapping",
            "agency_name",
            None,
            None,
            "Agency/name mapping consistency could not be computed.",
            str(agency_name_mapping_issues.iloc[0]["reason"]),
            "Re-run after scoped ingestion.",
        )
    elif not computable_mapping_rows.empty:
        add_issue(
            "review_required",
            "categorical_quality",
            "agency_name_mapping",
            "agency_name",
            int(computable_mapping_rows["affected_count"].sum()),
            percentage(
                int(computable_mapping_rows["affected_count"].sum()),
                total_row_count,
            ),
            "Agency codes map to multiple non-null agency-name labels.",
            "Human-readable agency metadata is not one-to-one.",
            "Review mapping history before any normalization.",
        )

for row in geographic_quality_summary.to_dict("records"):
    if row["severity"] == "not_computable":
        add_issue(
            "not_computable",
            "geographic_quality",
            row["check_name"],
            "incident_zip" if row["check_name"] == "incident_zip_malformed" else "",
            None,
            None,
            f"{row['check_name']} could not be computed through the current API query.",
            row["interpretation"],
            "Re-run after scoped ingestion without using a sample estimate.",
        )
        continue
    affected_count = safe_int(row["affected_count"])
    if affected_count and affected_count > 0:
        add_issue(
            row["severity"],
            "geographic_quality",
            row["check_name"],
            "",
            affected_count,
            safe_float(row["affected_percentage"]),
            f"{row['check_name']} detected in the complete source.",
            row["interpretation"],
            "Review geographic semantics during scoped ingestion; do not alter raw data.",
        )

failed_query_names_already_represented = {
    "exact_duplicate_records",
    "conflicting_duplicate_ids",
}
for failure in QUERY_FAILURES:
    if any(
        failure["query_name"].startswith(prefix)
        for prefix in failed_query_names_already_represented
    ):
        continue
    add_issue(
        "not_computable",
        "api_query_limitation",
        failure["query_name"],
        "",
        None,
        None,
        f"Full-dataset API query {failure['query_name']} failed.",
        failure["reason"],
        "Retry later or complete the check after reproducible scoped ingestion.",
    )

if not issue_rows:
    add_issue(
        "informational",
        "overall",
        "no_material_findings",
        "",
        0,
        0.0,
        "No non-zero or non-computable findings were detected.",
        "Computed checks found no registered issues.",
        "Proceed to Day 4 target-feasibility analysis.",
    )

data_quality_issues = pd.DataFrame(issue_rows)
data_quality_issues.insert(
    0,
    "issue_id",
    [f"DQ-{index:04d}" for index in range(1, len(data_quality_issues) + 1)],
)
data_quality_issues.to_csv(
    REPORT_TABLES_DIR / "data_quality_issues.csv",
    index=False,
)
data_quality_issues


,issue_id,severity,category,check_name,column_name,affected_count,affected_percentage,analysis_scope,issue,impact,recommended_next_action
0,DQ-0001,review_required,missingness,column_missing_values,taxi_company_borough,21909620,99.942155,full-dataset server-side aggregation,taxi_company_borough has high missingness.,Coverage should be considered in later scoped ingestion and analysis.,Assess operational meaning and scoped coverage before any treatment.
1,DQ-0002,review_required,missingness,column_missing_values,road_ramp,21871436,99.767976,full-dataset server-side aggregation,road_ramp has high missingness.,Coverage should be considered in later scoped ingestion and analysis.,Assess operational meaning and scoped coverage before any treatment.
2,DQ-0003,review_required,missingness,column_missing_values,bridge_highway_direction,21849184,99.666472,full-dataset server-side aggregation,bridge_highway_direction has high missingness.,Coverage should be considered in later scoped ingestion and analysis.,Assess operational meaning and scoped coverage before any treatment.
3,DQ-0004,critical_for_target,missingness,column_missing_values,due_date,21845314,99.648819,full-dataset server-side aggregation,due_date has high missingness.,Coverage limits rows that could be evaluated under the initial target concept; final eligibility is deferred to Day 4.,Assess operational meaning and scoped coverage before any treatment.
4,DQ-0005,review_required,missingness,column_missing_values,bridge_highway_name,21786553,99.380777,full-dataset server-side aggregation,bridge_highway_name has high missingness.,Coverage should be considered in later scoped ingestion and analysis.,Assess operational meaning and scoped coverage before any treatment.
5,DQ-0006,review_required,missingness,column_missing_values,bridge_highway_segment,21786538,99.380708,full-dataset server-side aggregation,bridge_highway_segment has high missingness.,Coverage should be considered in later scoped ingestion and analysis.,Assess operational meaning and scoped coverage before any treatment.
6,DQ-0007,review_required,missingness,column_missing_values,taxi_pick_up_location,21728808,99.117369,full-dataset server-side aggregation,taxi_pick_up_location has high missingness.,Coverage should be considered in later scoped ingestion and analysis.,Assess operational meaning and scoped coverage before any treatment.
7,DQ-0008,review_required,missingness,column_missing_values,vehicle_type,21480735,97.985768,full-dataset server-side aggregation,vehicle_type has high missingness.,Coverage should be considered in later scoped ingestion and analysis.,Assess operational meaning and scoped coverage before any treatment.
8,DQ-0009,review_required,missingness,column_missing_values,facility_type,20249222,92.368141,full-dataset server-side aggregation,facility_type has high missingness.,Coverage should be considered in later scoped ingestion and analysis.,Assess operational meaning and scoped coverage before any treatment.
9,DQ-0010,review_required,missingness,column_missing_values,descriptor_2,12286729,56.046712,full-dataset server-side aggregation,descriptor_2 has high missingness.,Coverage should be considered in later scoped ingestion and analysis.,Assess operational meaning and scoped coverage before any treatment.


## 16. Day 3 readiness decisions

> **Analysis scope:** Complete live NYC 311 dataset using server-side SoQL
> aggregation.

These decisions remain deliberately separate. Structural availability is not
the same as adequate target quality, and neither is the final feasibility
decision. Final target feasibility, eligibility, censoring, and scoped
thresholds belong to `03_target_feasibility.ipynb`.


In [16]:
core_missing = [
    column for column in CORE_REQUIRED_COLUMNS if column not in dataset_columns
]
essential_target_missing = [
    column for column in ESSENTIAL_TARGET_COLUMNS if column not in dataset_columns
]
essential_target_without_values = [
    column
    for column in ESSENTIAL_TARGET_COLUMNS
    if column in dataset_columns and non_null_counts.get(column, 0) == 0
]
candidate_missing = [
    column for column in CANDIDATE_FEATURE_COLUMNS if column not in dataset_columns
]

target_quality_findings = pd.concat(
    [
        missing_values_summary.loc[
            missing_values_summary["column_name"].isin(ESSENTIAL_TARGET_COLUMNS)
            & (missing_values_summary["missing_count"].fillna(0) > 0)
        ][["column_name", "missing_count"]],
        timestamp_quality_summary.loc[
            timestamp_quality_summary["severity"] == "critical_for_target",
            ["check_name", "affected_count"],
        ].rename(columns={"check_name": "column_name", "affected_count": "missing_count"}),
        status_timestamp_consistency.loc[
            status_timestamp_consistency["severity"] == "critical_for_target",
            ["check_name", "affected_count"],
        ].rename(columns={"check_name": "column_name", "affected_count": "missing_count"}),
    ],
    ignore_index=True,
)
has_target_quality_findings = bool(
    (pd.to_numeric(target_quality_findings["missing_count"], errors="coerce") > 0).any()
)

duplicate_computable = bool(
    (
        duplicate_summary.loc[
            duplicate_summary["check_name"].isin(
                ["null_unique_key", "duplicated_unique_key_groups"]
            ),
            "status",
        ]
        == "computed"
    ).all()
)
duplicate_integrity_passed = (
    duplicate_computable
    and null_unique_key_count == 0
    and duplicate_group_count == 0
)

chronology_violations = int(
    timestamp_quality_summary.loc[
        timestamp_quality_summary["severity"].isin(
            ["critical", "critical_for_target", "warning"]
        ),
        "affected_count",
    ].sum()
)

day3_readiness_summary = pd.DataFrame(
    [
        {
            "readiness_area": "Core schema readiness",
            "question": "Are the minimum complaint-level fields structurally available?",
            "status": "Passed" if not core_missing else "Failed",
            "evidence": (
                "All core fields are present."
                if not core_missing
                else "Missing: " + ", ".join(core_missing)
            ),
            "boundary": "Structural source availability only.",
        },
        {
            "readiness_area": "Target-input structural availability",
            "question": (
                "Are closed_date and due_date present and populated somewhere "
                "in the full source?"
            ),
            "status": (
                "Passed"
                if not essential_target_missing
                and not essential_target_without_values
                else "Failed"
            ),
            "evidence": (
                "Both essential fields exist and have at least one non-null value."
                if not essential_target_missing
                and not essential_target_without_values
                else (
                    "Missing or entirely null: "
                    + ", ".join(
                        essential_target_missing
                        + essential_target_without_values
                    )
                )
            ),
            "boundary": "Not a final target-feasibility decision.",
        },
        {
            "readiness_area": "Target-quality readiness",
            "question": (
                "Do target-related fields have sufficient coverage and "
                "consistent chronology?"
            ),
            "status": (
                "Provisional review required"
                if has_target_quality_findings
                else "No material issue detected by configured checks"
            ),
            "evidence": (
                "Non-zero target-related missingness or chronology findings exist."
                if has_target_quality_findings
                else "Configured complete-dataset checks found no material finding."
            ),
            "boundary": "Final thresholds and eligibility are deferred to Day 4.",
        },
        {
            "readiness_area": "Duplicate integrity",
            "question": "Is unique_key sufficiently unique for complaint-level analysis?",
            "status": (
                "Passed"
                if duplicate_integrity_passed
                else (
                    "Not computable"
                    if not duplicate_computable
                    else "Review required"
                )
            ),
            "evidence": (
                f"{null_unique_key_count:,} null identifiers; "
                f"{duplicate_group_count:,} duplicate identifier groups."
            ),
            "boundary": "Exact-row API limitations are reported separately.",
        },
        {
            "readiness_area": "Timestamp integrity",
            "question": "Are important event timestamps chronologically consistent?",
            "status": (
                "Passed" if chronology_violations == 0 else "Review required"
            ),
            "evidence": (
                f"{chronology_violations:,} affected occurrences summed across "
                "configured warning-or-higher chronology checks."
            ),
            "boundary": "Future due dates are informational, not automatic errors.",
        },
        {
            "readiness_area": "Candidate-feature availability",
            "question": (
                "Which potential creation-time fields are structurally and "
                "operationally available?"
            ),
            "status": (
                "Passed"
                if not candidate_missing
                else (
                    "Partially passed"
                    if len(candidate_missing) < len(CANDIDATE_FEATURE_COLUMNS)
                    else "Failed"
                )
            ),
            "evidence": (
                "Structurally available: "
                + ", ".join(
                    column
                    for column in CANDIDATE_FEATURE_COLUMNS
                    if column in dataset_columns
                )
                + (
                    "; missing: " + ", ".join(candidate_missing)
                    if candidate_missing
                    else ""
                )
            ),
            "boundary": (
                "Availability does not make a candidate a final model feature."
            ),
        },
    ]
)
day3_readiness_summary.to_csv(
    REPORT_TABLES_DIR / "day3_readiness_summary.csv",
    index=False,
)
day3_readiness_summary


,readiness_area,question,status,evidence,boundary
0,Core schema readiness,Are the minimum complaint-level fields structurally available?,Passed,All core fields are present.,Structural source availability only.
1,Target-input structural availability,Are closed_date and due_date present and populated somewhere in the full source?,Passed,Both essential fields exist and have at least one non-null value.,Not a final target-feasibility decision.
2,Target-quality readiness,Do target-related fields have sufficient coverage and consistent chronology?,Provisional review required,Non-zero target-related missingness or chronology findings exist.,Final thresholds and eligibility are deferred to Day 4.
3,Duplicate integrity,Is unique_key sufficiently unique for complaint-level analysis?,Passed,0 null identifiers; 0 duplicate identifier groups.,Exact-row API limitations are reported separately.
4,Timestamp integrity,Are important event timestamps chronologically consistent?,Review required,"545,471 affected occurrences summed across configured warning-or-higher chronology checks.","Future due dates are informational, not automatic errors."
5,Candidate-feature availability,Which potential creation-time fields are structurally and operationally available?,Passed,"Structurally available: agency, complaint_type, descriptor, borough, open_data_channel_type",Availability does not make a candidate a final model feature.


## 17. Query audit and generated data-quality report

> **Analysis scope:** Complete live NYC 311 dataset using server-side SoQL
> aggregation.

The report is assembled from computed notebook tables. It does not hardcode
dataset-specific counts and never calls an unsuccessful check a pass.


In [17]:
data_quality_query_audit = pd.DataFrame(
    QUERY_AUDIT_RECORDS,
    columns=[
        "query_name",
        "query",
        "execution_status",
        "retrieved_rows",
        "executed_at_utc",
        "error_message",
    ],
)
data_quality_query_audit.to_csv(
    REPORT_TABLES_DIR / "data_quality_query_audit.csv",
    index=False,
)


def markdown_table(frame: pd.DataFrame) -> str:
    """Render a dataframe as dependency-free Markdown."""
    display_frame = frame.copy()
    display_frame = display_frame.where(pd.notna(display_frame), "")
    headers = [str(column) for column in display_frame.columns]

    def clean(value: object) -> str:
        return str(value).replace("|", "\\|").replace("\n", " ")

    lines = [
        "| " + " | ".join(map(clean, headers)) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]
    lines.extend(
        "| " + " | ".join(clean(value) for value in row) + " |"
        for row in display_frame.itertuples(index=False, name=None)
    )
    return "\n".join(lines)


top_missing = missing_values_summary.head(15)[
    [
        "column_name",
        "missing_count",
        "missing_percentage",
        "missingness_level",
        "column_role",
    ]
]
api_limitations = data_quality_query_audit.loc[
    data_quality_query_audit["execution_status"] == "failed",
    ["query_name", "error_message"],
]
api_limitations_markdown = (
    markdown_table(api_limitations)
    if not api_limitations.empty
    else "No configured API query failed during this run."
)

quality_report = f"""# NYC 311 Data-Quality Report

Generated by `notebooks/02_schema_and_quality_analysis.ipynb` at
{analysis_timestamp_utc.isoformat()}.

## 1. Executive summary

This Day 3 analysis evaluated {total_row_count:,} live NYC 311 records through
server-side aggregation. No quality metric used a row-level sample. Readiness
decisions remain separate:

{markdown_table(day3_readiness_summary[['readiness_area', 'status', 'evidence', 'boundary']])}

## 2. Source and complete-dataset scope

{markdown_table(quality_analysis_scope)}

The source remains on NYC Open Data. The notebook retrieves aggregate and
grouped results only; it does not download the full row-level dataset.

## 3. Column roles

{markdown_table(column_role_summary[['column_name', 'column_group', 'role', 'required_for_current_stage', 'available_at_prediction_time', 'potential_leakage']])}

`closed_date` is the actual outcome timestamp and `due_date` is the expected
deadline. `status` supports eligibility and contradiction checks but is not
part of the mathematical comparison. Final status and other post-creation
values must not be used as prediction-time features. Candidate fields are not
guaranteed final model inputs.

## 4. Schema findings

{markdown_table(column_requirement_validation)}

Source metadata types, descriptions, and logical interpretations for all
configured fields are stored in `tables/schema_summary.csv`.

## 5. Missing-value findings

The highest missing percentages are:

{markdown_table(top_missing)}

A missing or entirely null `due_date` blocks the current target concept but
does not make the full NYC 311 dataset useless for unrelated descriptive
analysis.

## 6. Missing-pattern findings

{markdown_table(missing_pattern_summary)}

## 7. Duplicate findings

{markdown_table(duplicate_summary)}

All duplicate-ID groups are stored in
`tables/duplicate_unique_key_groups.csv`; conflicting identifiers are stored
in `tables/conflicting_duplicate_ids.csv`.

## 8. Timestamp-field findings

{markdown_table(timestamp_field_summary)}

Source-level format validity is enforced by the API's typed datetime schema;
downstream pandas parsing still requires validation after ingestion.

## 9. Timestamp-consistency findings

{markdown_table(timestamp_quality_summary)}

Future due dates are informational because open requests may legitimately have
future deadlines.

## 10. Status/timestamp findings

{markdown_table(status_timestamp_consistency)}

Status categories were derived transparently after inspecting the complete
source distribution. Final status is a leakage risk.

## 11. Categorical-quality findings

{markdown_table(categorical_quality_summary)}

Agency/name mapping issues and case-collision details are stored in their
dedicated tables. Placeholder-like values are review signals, not automatic
invalidity.

## 12. Geographic-quality findings

{markdown_table(geographic_quality_summary)}

The approximate NYC bounding box is informational and does not by itself prove
a record invalid.

## 13. Issue register

{markdown_table(data_quality_issues)}

## 14. Day 3 readiness decisions

{markdown_table(day3_readiness_summary)}

Target field structural availability is not final target feasibility.

## 15. API limitations and deferred checks

{api_limitations_markdown}

Any failed check has null counts and a `not_computable` issue. It is never
reported as zero. Expensive unsupported checks should be finalized after
reproducible scoped ingestion.

## 16. Implications for 03_target_feasibility.ipynb

Day 4 must define the scoped population, unresolved/censored-request handling,
target eligibility, provisional coverage thresholds, and whether `due_date`
exists at complaint creation. It must not use `closed_date`, final `status`, or
other post-creation values as prediction-time features. This notebook does not
create the target or declare final feasibility.
"""

QUALITY_REPORT_PATH.write_text(quality_report.strip() + "\n", encoding="utf-8")
display(Markdown(quality_report))


# NYC 311 Data-Quality Report

Generated by `notebooks/02_schema_and_quality_analysis.ipynb` at
2026-07-25T08:06:29.042177+00:00.

## 1. Executive summary

This Day 3 analysis evaluated 21,922,301 live NYC 311 records through
server-side aggregation. No quality metric used a row-level sample. Readiness
decisions remain separate:

| readiness_area | status | evidence | boundary |
| --- | --- | --- | --- |
| Core schema readiness | Passed | All core fields are present. | Structural source availability only. |
| Target-input structural availability | Passed | Both essential fields exist and have at least one non-null value. | Not a final target-feasibility decision. |
| Target-quality readiness | Provisional review required | Non-zero target-related missingness or chronology findings exist. | Final thresholds and eligibility are deferred to Day 4. |
| Duplicate integrity | Passed | 0 null identifiers; 0 duplicate identifier groups. | Exact-row API limitations are reported separately. |
| Timestamp integrity | Review required | 545,471 affected occurrences summed across configured warning-or-higher chronology checks. | Future due dates are informational, not automatic errors. |
| Candidate-feature availability | Passed | Structurally available: agency, complaint_type, descriptor, borough, open_data_channel_type | Availability does not make a candidate a final model feature. |

## 2. Source and complete-dataset scope

| source | api_endpoint | dataset_identifier | retrieval_timestamp_utc | total_dataset_rows | minimum_created_date | maximum_created_date | analysis_mode | sample_used_for_quality_metrics |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 311 Service Requests from 2020 to Present | https://data.cityofnewyork.us/resource/erm2-nwe9.json | erm2-nwe9 | 2026-07-25T08:06:29.042177+00:00 | 21922301 | 2020-01-01T00:00:00.000 | 2026-07-24T01:50:58.000 | full-dataset server-side aggregation | no |

The source remains on NYC Open Data. The notebook retrieves aggregate and
grouped results only; it does not download the full row-level dataset.

## 3. Column roles

| column_name | column_group | role | required_for_current_stage | available_at_prediction_time | potential_leakage |
| --- | --- | --- | --- | --- | --- |
| unique_key | core dataset requirement | identifier | yes | yes | no |
| created_date | core dataset requirement | event timestamp | yes | yes | no |
| closed_date | essential target input | actual outcome timestamp | yes, for target-input quality analysis | no | yes, outcome information |
| due_date | essential target input | expected resolution deadline | yes, for target-input quality analysis | must be confirmed | depends on whether it exists at complaint creation |
| status | target eligibility and validation | workflow state | yes, for target quality validation | initial state may be available; final state is not | yes, when final status is used |
| agency | core requirement and candidate feature | responsible organization | yes | yes | no |
| agency_name | supporting metadata | human-readable agency label | no | usually yes | no |
| complaint_type | core requirement and candidate feature | main request category | yes | yes | no |
| descriptor | candidate feature | detailed complaint category | no | generally yes; confirm source timing | no known leakage; validate later |
| borough | candidate feature | geographic grouping | no | generally yes; confirm source timing | no known leakage; validate later |
| open_data_channel_type | candidate feature | complaint intake channel | no | yes | no |

`closed_date` is the actual outcome timestamp and `due_date` is the expected
deadline. `status` supports eligibility and contradiction checks but is not
part of the mathematical comparison. Final status and other post-creation
values must not be used as prediction-time features. Candidate fields are not
guaranteed final model inputs.

## 4. Schema findings

| column_name | column_group | is_present | severity_if_missing | impact_if_missing |
| --- | --- | --- | --- | --- |
| unique_key | core_required | True | critical | The source cannot reliably support complaint-level analysis. |
| created_date | core_required | True | critical | The source cannot reliably support complaint-level analysis. |
| agency | core_required | True | critical | The source cannot reliably support complaint-level analysis. |
| complaint_type | core_required | True | critical | The source cannot reliably support complaint-level analysis. |
| closed_date | essential_target | True | critical_for_target | General NYC 311 analysis may remain possible, but the current missed-resolution-target definition cannot be evaluated. |
| due_date | essential_target | True | critical_for_target | General NYC 311 analysis may remain possible, but the current missed-resolution-target definition cannot be evaluated. |
| status | target_eligibility_and_validation | True | warning | Target eligibility and status/timestamp contradiction checks are limited. |
| agency | candidate_feature | True | warning | This specific future modelling or subgroup-analysis option is reduced. |
| complaint_type | candidate_feature | True | warning | This specific future modelling or subgroup-analysis option is reduced. |
| descriptor | candidate_feature | True | warning | This specific future modelling or subgroup-analysis option is reduced. |
| borough | candidate_feature | True | warning | This specific future modelling or subgroup-analysis option is reduced. |
| open_data_channel_type | candidate_feature | True | warning | This specific future modelling or subgroup-analysis option is reduced. |
| agency_name | supporting_metadata | True | informational | Reporting or consistency-check capability is reduced. |

Source metadata types, descriptions, and logical interpretations for all
configured fields are stored in `tables/schema_summary.csv`.

## 5. Missing-value findings

The highest missing percentages are:

| column_name | missing_count | missing_percentage | missingness_level | column_role |
| --- | --- | --- | --- | --- |
| taxi_company_borough | 21909620 | 99.942155 | high | selected_project_field |
| road_ramp | 21871436 | 99.767976 | high | selected_project_field |
| bridge_highway_direction | 21849184 | 99.666472 | high | selected_project_field |
| due_date | 21845314 | 99.648819 | high | essential_target |
| bridge_highway_name | 21786553 | 99.380777 | high | selected_project_field |
| bridge_highway_segment | 21786538 | 99.380708 | high | selected_project_field |
| taxi_pick_up_location | 21728808 | 99.117369 | high | selected_project_field |
| vehicle_type | 21480735 | 97.985768 | high | selected_project_field |
| facility_type | 20249222 | 92.368141 | high | selected_project_field |
| descriptor_2 | 12286729 | 56.046712 | high | selected_project_field |
| landmark | 9332651 | 42.571494 | high | selected_project_field |
| intersection_street_1 | 7702610 | 35.135956 | high | selected_project_field |
| intersection_street_2 | 7694667 | 35.099723 | high | selected_project_field |
| cross_street_2 | 6039204 | 27.548221 | moderate | selected_project_field |
| cross_street_1 | 6036857 | 27.537515 | moderate | selected_project_field |

A missing or entirely null `due_date` blocks the current target concept but
does not make the full NYC 311 dataset useless for unrelated descriptive
analysis.

## 6. Missing-pattern findings

| check_name | description | affected_count | affected_percentage | severity | interpretation |
| --- | --- | --- | --- | --- | --- |
| closed_and_due_both_missing | closed_date missing and due_date missing | 413762 | 1.887402 | critical_for_target | Neither essential input to the initial target concept is present. |
| closed_present_due_missing | closed_date present and due_date missing | 21431552 | 97.761417 | critical_for_target | An actual closure exists without the expected deadline. |
| due_present_closed_missing | due_date present and closed_date missing | 2031 | 0.009265 | review_required | May represent an open or unresolved complaint; eligibility is deferred. |
| closed_status_missing_closed_date | status indicates Closed and closed_date is missing | 53927 | 0.245992 | critical_for_target | Workflow state and outcome timestamp contradict each other. |
| non_closed_status_with_closed_date | observed non-Closed status with closed_date present | 78333 | 0.357321 | review_required | May be a workflow transition or a status/timestamp contradiction. |
| latitude_without_longitude | latitude present and longitude missing | 0 | 0.0 | warning | Coordinate pair is incomplete. |
| longitude_without_latitude | longitude present and latitude missing | 0 | 0.0 | warning | Coordinate pair is incomplete. |
| borough_and_zip_both_missing | borough missing and incident_zip missing | 38412 | 0.175219 | warning | Two common geographic grouping fields are absent together. |
| borough_missing_coordinates_present | borough missing while both coordinates are present | 18 | 8.2e-05 | review_required | Coordinates may permit later geovalidation without altering raw data. |

## 7. Duplicate findings

| check_name | analysis_scope | status | affected_count | affected_percentage | notes |
| --- | --- | --- | --- | --- | --- |
| null_unique_key | full-dataset server-side aggregation | computed | 0 | 0.0 | Rows where unique_key is null. |
| duplicated_unique_key_groups | full-dataset server-side aggregation | computed | 0 |  | Number of identifiers appearing more than once. |
| rows_in_duplicate_id_groups | full-dataset server-side aggregation | computed | 0 | 0.0 | Sum of record_count across duplicate-ID groups. |
| excess_duplicate_id_rows | full-dataset server-side aggregation | computed | 0 | 0.0 | Sum of record_count - 1 across duplicate-ID groups. |
| maximum_records_for_one_unique_key | full-dataset server-side aggregation | computed | 1 |  | Maximum source rows associated with one non-null identifier. |
| conflicting_duplicate_ids | full-dataset server-side aggregation | computed | 0 |  | No duplicate unique_key groups exist to conflict. |
| exact_duplicate_rows | full-dataset server-side aggregation | computed | 0 | 0.0 | All exact duplicate projection groups were retrieved. |
| excess_exact_duplicate_rows | full-dataset server-side aggregation | computed | 0 | 0.0 | All exact duplicate projection groups were retrieved. |

All duplicate-ID groups are stored in
`tables/duplicate_unique_key_groups.csv`; conflicting identifiers are stored
in `tables/conflicting_duplicate_ids.csv`.

## 8. Timestamp-field findings

| column_name | source_data_type | total_rows | non_null_count | missing_count | missing_percentage | minimum_timestamp | maximum_timestamp | source_type_valid | format_validity_note |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| created_date | calendar_date | 21922301 | 21922301 | 0 | 0.0 | 2020-01-01T00:00:00.000 | 2026-07-24T01:50:58.000 | True | Source-level format validity is enforced by the API's typed datetime schema; validate pandas parsing after ingestion. |
| closed_date | calendar_date | 21922301 | 21506508 | 415793 | 1.896667 | 1899-12-31T19:00:00.000 | 2033-03-01T00:00:00.000 | True | Source-level format validity is enforced by the API's typed datetime schema; validate pandas parsing after ingestion. |
| due_date | calendar_date | 21922301 | 76987 | 21845314 | 99.648819 | 2018-09-29T09:26:39.000 | 2026-08-22T20:02:47.000 | True | Source-level format validity is enforced by the API's typed datetime schema; validate pandas parsing after ingestion. |
| resolution_action_updated_date | calendar_date | 21922301 | 21742539 | 179762 | 0.819996 | 2019-07-18T00:00:00.000 | 2026-12-14T11:50:00.000 | True | Source-level format validity is enforced by the API's typed datetime schema; validate pandas parsing after ingestion. |

Source-level format validity is enforced by the API's typed datetime schema;
downstream pandas parsing still requires validation after ingestion.

## 9. Timestamp-consistency findings

| check_name | business_rule | affected_count | affected_percentage | severity | interpretation |
| --- | --- | --- | --- | --- | --- |
| closed_before_created | closed_date must not precede created_date | 46492 | 0.212076 | critical_for_target | Negative complaint lifetime undermines outcome chronology. |
| due_before_created | due_date should not precede created_date | 16 | 7.3e-05 | critical_for_target | Expected deadline predates the prediction moment. |
| resolution_update_before_created | resolution_action_updated_date should not precede created_date | 498960 | 2.276038 | warning | Recorded workflow update predates complaint creation. |
| created_after_analysis_time | created_date must not be later than extraction time | 0 | 0.0 | warning | Creation timestamp is in the future relative to extraction. |
| closed_after_analysis_time | closed_date must not be later than extraction time | 2 | 9e-06 | warning | Outcome timestamp is in the future relative to extraction. |
| resolution_update_after_analysis_time | resolution_action_updated_date must not be later than extraction time | 1 | 5e-06 | warning | Workflow update is in the future relative to extraction. |
| due_after_analysis_time | Future due dates are permitted and should be reviewed with status | 1116 | 0.005091 | informational | May be a legitimate future SLA deadline for an open complaint. |

Future due dates are informational because open requests may legitimately have
future deadlines.

## 10. Status/timestamp findings

| check_name | business_rule | affected_count | affected_percentage | severity | interpretation |
| --- | --- | --- | --- | --- | --- |
| closed_status_missing_closed_date | Closed status should have a closed_date | 53927 | 0.245992 | critical_for_target | Closure status lacks its outcome timestamp. |
| active_status_with_closed_date | Open or active status is expected to lack final closure | 75646 | 0.345064 | review_required | May reflect a reopened complaint or a workflow inconsistency. |
| closed_status_closed_before_created | Closed complaints must not close before creation | 1012 | 0.004616 | critical_for_target | Closed workflow state has impossible outcome chronology. |
| special_status_with_closed_date | Special or excluded statuses require explicit eligibility review | 2687 | 0.012257 | review_required | Closure information exists for a special-status complaint. |

Status categories were derived transparently after inspecting the complete
source distribution. Final status is a leakage risk.

## 11. Categorical-quality findings

| column_name | check_name | analysis_scope | status | affected_count | affected_percentage | interpretation |
| --- | --- | --- | --- | --- | --- | --- |
| agency | null_values | full-dataset server-side aggregation | computed | 0 | 0.0 | Null source values in the complete grouped distribution. |
| agency | empty_strings | full-dataset server-side aggregation | computed | 0 | 0.0 | Present-but-empty strings; not the same as API nulls. |
| agency | leading_or_trailing_whitespace | full-dataset server-side aggregation | computed | 0 | 0.0 | Raw category labels with surrounding whitespace. |
| agency | placeholder_values | full-dataset server-side aggregation | computed | 0 | 0.0 | Configured placeholder-like labels reported for review, not rejection. |
| agency | case_insensitive_collisions | full-dataset server-side aggregation | computed | 0 | 0.0 | Rows in lowercased category groups containing multiple raw variants. |
| agency_name | null_values | full-dataset server-side aggregation | computed | 0 | 0.0 | Null source values in the complete grouped distribution. |
| agency_name | empty_strings | full-dataset server-side aggregation | computed | 0 | 0.0 | Present-but-empty strings; not the same as API nulls. |
| agency_name | leading_or_trailing_whitespace | full-dataset server-side aggregation | computed | 0 | 0.0 | Raw category labels with surrounding whitespace. |
| agency_name | placeholder_values | full-dataset server-side aggregation | computed | 0 | 0.0 | Configured placeholder-like labels reported for review, not rejection. |
| agency_name | case_insensitive_collisions | full-dataset server-side aggregation | computed | 0 | 0.0 | Rows in lowercased category groups containing multiple raw variants. |
| complaint_type | null_values | full-dataset server-side aggregation | computed | 0 | 0.0 | Null source values in the complete grouped distribution. |
| complaint_type | empty_strings | full-dataset server-side aggregation | computed | 0 | 0.0 | Present-but-empty strings; not the same as API nulls. |
| complaint_type | leading_or_trailing_whitespace | full-dataset server-side aggregation | computed | 0 | 0.0 | Raw category labels with surrounding whitespace. |
| complaint_type | placeholder_values | full-dataset server-side aggregation | computed | 42 | 0.000192 | Configured placeholder-like labels reported for review, not rejection. |
| complaint_type | case_insensitive_collisions | full-dataset server-side aggregation | computed | 4504551 | 20.547802 | Rows in lowercased category groups containing multiple raw variants. |
| descriptor | null_values | full-dataset server-side aggregation | computed | 158566 | 0.723309 | Null source values in the complete grouped distribution. |
| descriptor | empty_strings | full-dataset server-side aggregation | computed | 0 | 0.0 | Present-but-empty strings; not the same as API nulls. |
| descriptor | leading_or_trailing_whitespace | full-dataset server-side aggregation | computed | 0 | 0.0 | Raw category labels with surrounding whitespace. |
| descriptor | placeholder_values | full-dataset server-side aggregation | computed | 651328 | 2.971075 | Configured placeholder-like labels reported for review, not rejection. |
| descriptor | case_insensitive_collisions | full-dataset server-side aggregation | computed | 4133458 | 18.855037 | Rows in lowercased category groups containing multiple raw variants. |
| status | null_values | full-dataset server-side aggregation | computed | 0 | 0.0 | Null source values in the complete grouped distribution. |
| status | empty_strings | full-dataset server-side aggregation | computed | 0 | 0.0 | Present-but-empty strings; not the same as API nulls. |
| status | leading_or_trailing_whitespace | full-dataset server-side aggregation | computed | 0 | 0.0 | Raw category labels with surrounding whitespace. |
| status | placeholder_values | full-dataset server-side aggregation | computed | 2782 | 0.01269 | Configured placeholder-like labels reported for review, not rejection. |
| status | case_insensitive_collisions | full-dataset server-side aggregation | computed | 0 | 0.0 | Rows in lowercased category groups containing multiple raw variants. |
| borough | null_values | full-dataset server-side aggregation | computed | 38433 | 0.175315 | Null source values in the complete grouped distribution. |
| borough | empty_strings | full-dataset server-side aggregation | computed | 0 | 0.0 | Present-but-empty strings; not the same as API nulls. |
| borough | leading_or_trailing_whitespace | full-dataset server-side aggregation | computed | 0 | 0.0 | Raw category labels with surrounding whitespace. |
| borough | placeholder_values | full-dataset server-side aggregation | computed | 40043 | 0.182659 | Configured placeholder-like labels reported for review, not rejection. |
| borough | case_insensitive_collisions | full-dataset server-side aggregation | computed | 0 | 0.0 | Rows in lowercased category groups containing multiple raw variants. |
| open_data_channel_type | null_values | full-dataset server-side aggregation | computed | 0 | 0.0 | Null source values in the complete grouped distribution. |
| open_data_channel_type | empty_strings | full-dataset server-side aggregation | computed | 0 | 0.0 | Present-but-empty strings; not the same as API nulls. |
| open_data_channel_type | leading_or_trailing_whitespace | full-dataset server-side aggregation | computed | 0 | 0.0 | Raw category labels with surrounding whitespace. |
| open_data_channel_type | placeholder_values | full-dataset server-side aggregation | computed | 1887614 | 8.610474 | Configured placeholder-like labels reported for review, not rejection. |
| open_data_channel_type | case_insensitive_collisions | full-dataset server-side aggregation | computed | 0 | 0.0 | Rows in lowercased category groups containing multiple raw variants. |

Agency/name mapping issues and case-collision details are stored in their
dedicated tables. Placeholder-like values are review signals, not automatic
invalidity.

## 12. Geographic-quality findings

| check_name | affected_count | affected_percentage | severity | interpretation |
| --- | --- | --- | --- | --- |
| latitude_outside_global_range | 0 | 0.0 | warning | Latitude is outside the mathematically valid global range. |
| longitude_outside_global_range | 0 | 0.0 | warning | Longitude is outside the mathematically valid global range. |
| latitude_without_longitude | 0 | 0.0 | warning | Coordinate pair is incomplete. |
| longitude_without_latitude | 0 | 0.0 | warning | Coordinate pair is incomplete. |
| both_coordinates_missing | 410402 | 1.872075 | informational | No coordinate pair is available for geographic modelling. |
| borough_missing_coordinates_present | 18 | 8.2e-05 | review_required | Coordinates exist while borough grouping is absent. |
| coordinates_outside_approximate_nyc_box | 1 | 5e-06 | informational | Outside a broad approximate NYC box; inspect context before judging validity. |
| incident_zip_malformed | 24 | 0.000109 | review_required | ZIP is not exactly five ASCII digits. |

The approximate NYC bounding box is informational and does not by itself prove
a record invalid.

## 13. Issue register

| issue_id | severity | category | check_name | column_name | affected_count | affected_percentage | analysis_scope | issue | impact | recommended_next_action |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| DQ-0001 | review_required | missingness | column_missing_values | taxi_company_borough | 21909620 | 99.942155 | full-dataset server-side aggregation | taxi_company_borough has high missingness. | Coverage should be considered in later scoped ingestion and analysis. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0002 | review_required | missingness | column_missing_values | road_ramp | 21871436 | 99.767976 | full-dataset server-side aggregation | road_ramp has high missingness. | Coverage should be considered in later scoped ingestion and analysis. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0003 | review_required | missingness | column_missing_values | bridge_highway_direction | 21849184 | 99.666472 | full-dataset server-side aggregation | bridge_highway_direction has high missingness. | Coverage should be considered in later scoped ingestion and analysis. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0004 | critical_for_target | missingness | column_missing_values | due_date | 21845314 | 99.648819 | full-dataset server-side aggregation | due_date has high missingness. | Coverage limits rows that could be evaluated under the initial target concept; final eligibility is deferred to Day 4. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0005 | review_required | missingness | column_missing_values | bridge_highway_name | 21786553 | 99.380777 | full-dataset server-side aggregation | bridge_highway_name has high missingness. | Coverage should be considered in later scoped ingestion and analysis. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0006 | review_required | missingness | column_missing_values | bridge_highway_segment | 21786538 | 99.380708 | full-dataset server-side aggregation | bridge_highway_segment has high missingness. | Coverage should be considered in later scoped ingestion and analysis. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0007 | review_required | missingness | column_missing_values | taxi_pick_up_location | 21728808 | 99.117369 | full-dataset server-side aggregation | taxi_pick_up_location has high missingness. | Coverage should be considered in later scoped ingestion and analysis. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0008 | review_required | missingness | column_missing_values | vehicle_type | 21480735 | 97.985768 | full-dataset server-side aggregation | vehicle_type has high missingness. | Coverage should be considered in later scoped ingestion and analysis. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0009 | review_required | missingness | column_missing_values | facility_type | 20249222 | 92.368141 | full-dataset server-side aggregation | facility_type has high missingness. | Coverage should be considered in later scoped ingestion and analysis. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0010 | review_required | missingness | column_missing_values | descriptor_2 | 12286729 | 56.046712 | full-dataset server-side aggregation | descriptor_2 has high missingness. | Coverage should be considered in later scoped ingestion and analysis. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0011 | review_required | missingness | column_missing_values | landmark | 9332651 | 42.571494 | full-dataset server-side aggregation | landmark has high missingness. | Coverage should be considered in later scoped ingestion and analysis. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0012 | review_required | missingness | column_missing_values | intersection_street_1 | 7702610 | 35.135956 | full-dataset server-side aggregation | intersection_street_1 has high missingness. | Coverage should be considered in later scoped ingestion and analysis. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0013 | review_required | missingness | column_missing_values | intersection_street_2 | 7694667 | 35.099723 | full-dataset server-side aggregation | intersection_street_2 has high missingness. | Coverage should be considered in later scoped ingestion and analysis. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0014 | informational | missingness | column_missing_values | cross_street_2 | 6039204 | 27.548221 | full-dataset server-side aggregation | cross_street_2 has moderate missingness. | Coverage should be considered in later scoped ingestion and analysis. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0015 | informational | missingness | column_missing_values | cross_street_1 | 6036857 | 27.537515 | full-dataset server-side aggregation | cross_street_1 has moderate missingness. | Coverage should be considered in later scoped ingestion and analysis. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0016 | informational | missingness | column_missing_values | location_type | 2913577 | 13.290471 | full-dataset server-side aggregation | location_type has moderate missingness. | Coverage should be considered in later scoped ingestion and analysis. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0017 | informational | missingness | column_missing_values | address_type | 2796728 | 12.757456 | full-dataset server-side aggregation | address_type has moderate missingness. | Coverage should be considered in later scoped ingestion and analysis. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0018 | informational | missingness | column_missing_values | bbl | 2565306 | 11.70181 | full-dataset server-side aggregation | bbl has moderate missingness. | Coverage should be considered in later scoped ingestion and analysis. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0019 | informational | missingness | column_missing_values | city | 1135277 | 5.17864 | full-dataset server-side aggregation | city has moderate missingness. | Coverage should be considered in later scoped ingestion and analysis. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0020 | informational | missingness | column_missing_values | street_name | 921170 | 4.201977 | full-dataset server-side aggregation | street_name has low missingness. | Coverage should be considered in later scoped ingestion and analysis. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0021 | informational | missingness | column_missing_values | incident_address | 920489 | 4.19887 | full-dataset server-side aggregation | incident_address has low missingness. | Coverage should be considered in later scoped ingestion and analysis. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0022 | informational | missingness | column_missing_values | resolution_description | 775819 | 3.538949 | full-dataset server-side aggregation | resolution_description has low missingness. | Coverage should be considered in later scoped ingestion and analysis. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0023 | informational | missingness | column_missing_values | council_district | 518991 | 2.367411 | full-dataset server-side aggregation | council_district has low missingness. | Coverage should be considered in later scoped ingestion and analysis. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0024 | critical_for_target | missingness | column_missing_values | closed_date | 415793 | 1.896667 | full-dataset server-side aggregation | closed_date has low missingness. | Coverage limits rows that could be evaluated under the initial target concept; final eligibility is deferred to Day 4. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0025 | informational | missingness | column_missing_values | location | 410403 | 1.87208 | full-dataset server-side aggregation | location has low missingness. | Coverage should be considered in later scoped ingestion and analysis. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0026 | informational | missingness | column_missing_values | latitude | 410402 | 1.872075 | full-dataset server-side aggregation | latitude has low missingness. | Coverage should be considered in later scoped ingestion and analysis. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0027 | informational | missingness | column_missing_values | longitude | 410402 | 1.872075 | full-dataset server-side aggregation | longitude has low missingness. | Coverage should be considered in later scoped ingestion and analysis. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0028 | informational | missingness | column_missing_values | x_coordinate_state_plane | 410211 | 1.871204 | full-dataset server-side aggregation | x_coordinate_state_plane has low missingness. | Coverage should be considered in later scoped ingestion and analysis. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0029 | informational | missingness | column_missing_values | y_coordinate_state_plane | 408727 | 1.864435 | full-dataset server-side aggregation | y_coordinate_state_plane has low missingness. | Coverage should be considered in later scoped ingestion and analysis. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0030 | informational | missingness | column_missing_values | incident_zip | 310459 | 1.416179 | full-dataset server-side aggregation | incident_zip has low missingness. | Coverage should be considered in later scoped ingestion and analysis. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0031 | informational | missingness | column_missing_values | resolution_action_updated_date | 179762 | 0.819996 | full-dataset server-side aggregation | resolution_action_updated_date has low missingness. | Coverage should be considered in later scoped ingestion and analysis. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0032 | informational | missingness | column_missing_values | descriptor | 158566 | 0.723309 | full-dataset server-side aggregation | descriptor has low missingness. | Coverage limits this candidate's future usefulness but does not invalidate the core dataset. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0033 | informational | missingness | column_missing_values | borough | 38433 | 0.175315 | full-dataset server-side aggregation | borough has low missingness. | Coverage limits this candidate's future usefulness but does not invalidate the core dataset. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0034 | informational | missingness | column_missing_values | community_board | 38433 | 0.175315 | full-dataset server-side aggregation | community_board has low missingness. | Coverage should be considered in later scoped ingestion and analysis. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0035 | informational | missingness | column_missing_values | park_borough | 38433 | 0.175315 | full-dataset server-side aggregation | park_borough has low missingness. | Coverage should be considered in later scoped ingestion and analysis. | Assess operational meaning and scoped coverage before any treatment. |
| DQ-0036 | critical_for_target | missing_patterns | closed_and_due_both_missing |  | 413762 | 1.887402 | full-dataset server-side aggregation | closed_date missing and due_date missing | Neither essential input to the initial target concept is present. | Define scoped eligibility or validation handling in Day 4. |
| DQ-0037 | critical_for_target | missing_patterns | closed_present_due_missing |  | 21431552 | 97.761417 | full-dataset server-side aggregation | closed_date present and due_date missing | An actual closure exists without the expected deadline. | Define scoped eligibility or validation handling in Day 4. |
| DQ-0038 | review_required | missing_patterns | due_present_closed_missing |  | 2031 | 0.009265 | full-dataset server-side aggregation | due_date present and closed_date missing | May represent an open or unresolved complaint; eligibility is deferred. | Define scoped eligibility or validation handling in Day 4. |
| DQ-0039 | critical_for_target | missing_patterns | closed_status_missing_closed_date |  | 53927 | 0.245992 | full-dataset server-side aggregation | status indicates Closed and closed_date is missing | Workflow state and outcome timestamp contradict each other. | Define scoped eligibility or validation handling in Day 4. |
| DQ-0040 | review_required | missing_patterns | non_closed_status_with_closed_date |  | 78333 | 0.357321 | full-dataset server-side aggregation | observed non-Closed status with closed_date present | May be a workflow transition or a status/timestamp contradiction. | Define scoped eligibility or validation handling in Day 4. |
| DQ-0041 | warning | missing_patterns | borough_and_zip_both_missing |  | 38412 | 0.175219 | full-dataset server-side aggregation | borough missing and incident_zip missing | Two common geographic grouping fields are absent together. | Define scoped eligibility or validation handling in Day 4. |
| DQ-0042 | review_required | missing_patterns | borough_missing_coordinates_present |  | 18 | 8.2e-05 | full-dataset server-side aggregation | borough missing while both coordinates are present | Coordinates may permit later geovalidation without altering raw data. | Define scoped eligibility or validation handling in Day 4. |
| DQ-0043 | critical_for_target | timestamp_consistency | closed_before_created |  | 46492 | 0.212076 | full-dataset server-side aggregation | closed_date must not precede created_date | Negative complaint lifetime undermines outcome chronology. | Review scope and eligibility implications in Day 4. |
| DQ-0044 | critical_for_target | timestamp_consistency | due_before_created |  | 16 | 7.3e-05 | full-dataset server-side aggregation | due_date should not precede created_date | Expected deadline predates the prediction moment. | Review scope and eligibility implications in Day 4. |
| DQ-0045 | warning | timestamp_consistency | resolution_update_before_created |  | 498960 | 2.276038 | full-dataset server-side aggregation | resolution_action_updated_date should not precede created_date | Recorded workflow update predates complaint creation. | Review scope and eligibility implications in Day 4. |
| DQ-0046 | warning | timestamp_consistency | closed_after_analysis_time |  | 2 | 9e-06 | full-dataset server-side aggregation | closed_date must not be later than extraction time | Outcome timestamp is in the future relative to extraction. | Review scope and eligibility implications in Day 4. |
| DQ-0047 | warning | timestamp_consistency | resolution_update_after_analysis_time |  | 1 | 5e-06 | full-dataset server-side aggregation | resolution_action_updated_date must not be later than extraction time | Workflow update is in the future relative to extraction. | Review scope and eligibility implications in Day 4. |
| DQ-0048 | informational | timestamp_consistency | due_after_analysis_time |  | 1116 | 0.005091 | full-dataset server-side aggregation | Future due dates are permitted and should be reviewed with status | May be a legitimate future SLA deadline for an open complaint. | Review scope and eligibility implications in Day 4. |
| DQ-0049 | critical_for_target | status_timestamp_consistency | closed_status_missing_closed_date |  | 53927 | 0.245992 | full-dataset server-side aggregation | Closed status should have a closed_date | Closure status lacks its outcome timestamp. | Review scope and eligibility implications in Day 4. |
| DQ-0050 | review_required | status_timestamp_consistency | active_status_with_closed_date |  | 75646 | 0.345064 | full-dataset server-side aggregation | Open or active status is expected to lack final closure | May reflect a reopened complaint or a workflow inconsistency. | Review scope and eligibility implications in Day 4. |
| DQ-0051 | critical_for_target | status_timestamp_consistency | closed_status_closed_before_created |  | 1012 | 0.004616 | full-dataset server-side aggregation | Closed complaints must not close before creation | Closed workflow state has impossible outcome chronology. | Review scope and eligibility implications in Day 4. |
| DQ-0052 | review_required | status_timestamp_consistency | special_status_with_closed_date |  | 2687 | 0.012257 | full-dataset server-side aggregation | Special or excluded statuses require explicit eligibility review | Closure information exists for a special-status complaint. | Review scope and eligibility implications in Day 4. |
| DQ-0053 | review_required | categorical_quality | placeholder_values | complaint_type | 42 | 0.000192 | full-dataset server-side aggregation | placeholder_values detected for complaint_type. | Configured placeholder-like labels reported for review, not rejection. | Review category semantics before defining any normalization rule. |
| DQ-0054 | review_required | categorical_quality | case_insensitive_collisions | complaint_type | 4504551 | 20.547802 | full-dataset server-side aggregation | case_insensitive_collisions detected for complaint_type. | Rows in lowercased category groups containing multiple raw variants. | Review category semantics before defining any normalization rule. |
| DQ-0055 | review_required | categorical_quality | null_values | descriptor | 158566 | 0.723309 | full-dataset server-side aggregation | null_values detected for descriptor. | Null source values in the complete grouped distribution. | Review category semantics before defining any normalization rule. |
| DQ-0056 | review_required | categorical_quality | placeholder_values | descriptor | 651328 | 2.971075 | full-dataset server-side aggregation | placeholder_values detected for descriptor. | Configured placeholder-like labels reported for review, not rejection. | Review category semantics before defining any normalization rule. |
| DQ-0057 | review_required | categorical_quality | case_insensitive_collisions | descriptor | 4133458 | 18.855037 | full-dataset server-side aggregation | case_insensitive_collisions detected for descriptor. | Rows in lowercased category groups containing multiple raw variants. | Review category semantics before defining any normalization rule. |
| DQ-0058 | review_required | categorical_quality | placeholder_values | status | 2782 | 0.01269 | full-dataset server-side aggregation | placeholder_values detected for status. | Configured placeholder-like labels reported for review, not rejection. | Review category semantics before defining any normalization rule. |
| DQ-0059 | review_required | categorical_quality | null_values | borough | 38433 | 0.175315 | full-dataset server-side aggregation | null_values detected for borough. | Null source values in the complete grouped distribution. | Review category semantics before defining any normalization rule. |
| DQ-0060 | review_required | categorical_quality | placeholder_values | borough | 40043 | 0.182659 | full-dataset server-side aggregation | placeholder_values detected for borough. | Configured placeholder-like labels reported for review, not rejection. | Review category semantics before defining any normalization rule. |
| DQ-0061 | review_required | categorical_quality | placeholder_values | open_data_channel_type | 1887614 | 8.610474 | full-dataset server-side aggregation | placeholder_values detected for open_data_channel_type. | Configured placeholder-like labels reported for review, not rejection. | Review category semantics before defining any normalization rule. |
| DQ-0062 | review_required | categorical_quality | agency_name_mapping | agency_name | 288706 | 1.316951 | full-dataset server-side aggregation | Agency codes map to multiple non-null agency-name labels. | Human-readable agency metadata is not one-to-one. | Review mapping history before any normalization. |
| DQ-0063 | informational | geographic_quality | both_coordinates_missing |  | 410402 | 1.872075 | full-dataset server-side aggregation | both_coordinates_missing detected in the complete source. | No coordinate pair is available for geographic modelling. | Review geographic semantics during scoped ingestion; do not alter raw data. |
| DQ-0064 | review_required | geographic_quality | borough_missing_coordinates_present |  | 18 | 8.2e-05 | full-dataset server-side aggregation | borough_missing_coordinates_present detected in the complete source. | Coordinates exist while borough grouping is absent. | Review geographic semantics during scoped ingestion; do not alter raw data. |
| DQ-0065 | informational | geographic_quality | coordinates_outside_approximate_nyc_box |  | 1 | 5e-06 | full-dataset server-side aggregation | coordinates_outside_approximate_nyc_box detected in the complete source. | Outside a broad approximate NYC box; inspect context before judging validity. | Review geographic semantics during scoped ingestion; do not alter raw data. |
| DQ-0066 | review_required | geographic_quality | incident_zip_malformed |  | 24 | 0.000109 | full-dataset server-side aggregation | incident_zip_malformed detected in the complete source. | ZIP is not exactly five ASCII digits. | Review geographic semantics during scoped ingestion; do not alter raw data. |

## 14. Day 3 readiness decisions

| readiness_area | question | status | evidence | boundary |
| --- | --- | --- | --- | --- |
| Core schema readiness | Are the minimum complaint-level fields structurally available? | Passed | All core fields are present. | Structural source availability only. |
| Target-input structural availability | Are closed_date and due_date present and populated somewhere in the full source? | Passed | Both essential fields exist and have at least one non-null value. | Not a final target-feasibility decision. |
| Target-quality readiness | Do target-related fields have sufficient coverage and consistent chronology? | Provisional review required | Non-zero target-related missingness or chronology findings exist. | Final thresholds and eligibility are deferred to Day 4. |
| Duplicate integrity | Is unique_key sufficiently unique for complaint-level analysis? | Passed | 0 null identifiers; 0 duplicate identifier groups. | Exact-row API limitations are reported separately. |
| Timestamp integrity | Are important event timestamps chronologically consistent? | Review required | 545,471 affected occurrences summed across configured warning-or-higher chronology checks. | Future due dates are informational, not automatic errors. |
| Candidate-feature availability | Which potential creation-time fields are structurally and operationally available? | Passed | Structurally available: agency, complaint_type, descriptor, borough, open_data_channel_type | Availability does not make a candidate a final model feature. |

Target field structural availability is not final target feasibility.

## 15. API limitations and deferred checks

No configured API query failed during this run.

Any failed check has null counts and a `not_computable` issue. It is never
reported as zero. Expensive unsupported checks should be finalized after
reproducible scoped ingestion.

## 16. Implications for 03_target_feasibility.ipynb

Day 4 must define the scoped population, unresolved/censored-request handling,
target eligibility, provisional coverage thresholds, and whether `due_date`
exists at complaint creation. It must not use `closed_date`, final `status`, or
other post-creation values as prediction-time features. This notebook does not
create the target or declare final feasibility.


## 18. Day 3 completion boundary

> **Analysis scope:** Complete live NYC 311 dataset using server-side SoQL
> aggregation.

This notebook validates and reports the complete live source without modifying
it. It performs no cleaning, imputation, row removal, target creation, final
feature selection, splitting, or modelling.

Material findings use complete-dataset server-side evidence. Any example-row
query would be inspection-only and explicitly labelled; this implementation
does not require example rows for its totals or decisions.

The next step is `notebooks/03_target_feasibility.ipynb`, where final target
feasibility and eligibility rules must be assessed.
